# HIVE-Bic: selection-honest biclustering of multi-tissue transcriptomes

Honest Inference Via thinned Evidence. This notebook reproduces the full study on the free Colab tier (2 CPUs, no GPU):

1. download the pig tissue atlas E-MTAB-5895, NCBI orthologs, GO sets and GTEx v10;
2. prepare the 2,000-gene ortholog matrix;
3. run HIVE-Bic on one semi-synthetic matrix (about 1 minute);
4. run the benchmark (main, null, ablation, sensitivity), the real-data study and the statistics;
5. draw the figures.

Set `HIVE_SEEDS` below to trade run time for precision: 12 seeds reproduce the paper (several hours on 2 CPUs); 3 seeds give a quick check.

In [ ]:
!pip -q install --no-deps biclustlib
!pip -q install bottleneck huggingface_hub pyarrow scikit-learn scipy pandas matplotlib
import os
os.makedirs("hive", exist_ok=True); os.chdir("hive")
os.environ["HIVE_DATA"] = os.path.abspath("data"); os.environ["HIVE_OUT"] = os.path.abspath("results"); os.environ["HIVE_FIGS"] = os.path.abspath("figs")
os.environ["HIVE_SEEDS"] = "3"   # 12 for the paper
for d in ("data", "results", "figs"): os.makedirs(d, exist_ok=True)

## 1. Source code
Each cell writes one module of the package.

In [ ]:
%%writefile hive.py
"""
HIVE-Bic: Honest Inference Via thinned Evidence for Biclustering.

Pipeline
  1. dispersion-conditional negative-binomial data thinning  -> discovery fold X1, confirmation fold X2
  2. diversity-constrained binary bee swarm on the discovery fold (null-corrected coherence objective)
  3. cross-fitted LogitBoost membership refinement (genes and conditions)
  4. certification on the confirmation fold (row/column permutation nulls, Benjamini-Hochberg)
All randomness is driven by numpy Generators so that every run is reproducible from a seed.
"""
import numpy as np
from scipy.special import expit
from sklearn.tree import DecisionTreeRegressor

# ----------------------------------------------------------------------------------------------
# 1. Dispersion-conditional NB thinning
# ----------------------------------------------------------------------------------------------
def size_factors(lib):
    lib = np.asarray(lib, float)
    return lib / np.exp(np.mean(np.log(lib)))


def estimate_dispersion(X, lib, groups, floor=1e-3, cap=5.0, shrink=0.5):
    """Method-of-moments NB dispersion pooled over replicate groups (conditions with >=2 samples).
    Var(y_gj) = mu_g/s_j + phi_g mu_g^2 for size-factor normalised y = x/s.  Estimates are shrunk
    towards the median on the log scale to stabilise genes with few replicate groups."""
    s = size_factors(lib)
    Y = X / s[None, :]
    num = np.zeros(X.shape[0]); den = np.zeros(X.shape[0])
    for gname in np.unique(groups):
        idx = np.where(groups == gname)[0]
        if len(idx) < 2:
            continue
        m = Y[:, idx].mean(1); v = Y[:, idx].var(1, ddof=1)
        inv_s = np.mean(1.0 / s[idx])
        ok = m > 0
        est = np.zeros_like(m); est[ok] = (v[ok] - m[ok] * inv_s) / m[ok] ** 2
        num += (len(idx) - 1) * est; den += (len(idx) - 1)
    phi = np.clip(num / np.maximum(den, 1), floor, cap)
    lphi = np.log(phi); med = np.median(lphi)
    return np.exp(shrink * lphi + (1 - shrink) * med)


def nb_thin(X, phi, eps, rng):
    """Negative-binomial data thinning: X1 | X ~ BetaBinomial(X, eps*b, (1-eps)*b), b = 1/phi.
    Under X ~ NB(mu, b) this gives independent X1 ~ NB(eps*mu, eps*b) and X2 = X - X1."""
    b = (1.0 / phi)[:, None] * np.ones((1, X.shape[1]))
    p = rng.beta(eps * b, (1 - eps) * b)
    X1 = rng.binomial(X.astype(np.int64), p)
    return X1, X - X1


def log_cpm(X, lib):
    return np.log2(X / np.asarray(lib, float)[None, :] * 1e6 + 1.0)


# ----------------------------------------------------------------------------------------------
# 2. Null-corrected coherence objective
# ----------------------------------------------------------------------------------------------
def _stdrows(Z):
    Z = Z - Z.mean(1, keepdims=True)
    sd = Z.std(1, keepdims=True); sd[sd < 1e-9] = 1.0
    return Z / sd


def standardize(Y):
    """Robust row standardisation (median / MAD): a condition set J that covers a minority of the
    conditions does not pull the centre of its member genes, so cells outside J stay near zero."""
    Y = np.asarray(Y, float)
    med = np.median(Y, 1, keepdims=True)
    mad = 1.4826 * np.median(np.abs(Y - med), 1, keepdims=True)
    sd = Y.std(1, keepdims=True)
    scale = np.maximum(mad, 0.25 * sd); scale[scale < 1e-9] = 1.0
    return np.clip((Y - med) / scale, -8, 8)


def coherence(Z, I, J):
    """Null-corrected rank-one energy of the submatrix Z[I, J] of the row-standardised matrix Z.
    lambda_1 / (|I||J|) is the share of squared signal carried by one gene loading x condition profile;
    (|I|^-1/2 + |J|^-1/2)^2 is its value for a pure-noise block (random-matrix edge), so the difference
    is positive only for structure beyond chance.  Shifting, scaling and sign-flipped (anti-correlated)
    members all load on the same singular pair, so one statistic covers the three pattern families."""
    I = np.asarray(I); J = np.asarray(J)
    if len(I) < 3 or len(J) < 3:
        return -1.0
    S = Z[np.ix_(I, J)]
    G = S.T @ S if len(J) <= len(I) else S @ S.T
    lam = np.linalg.eigvalsh(G)[-1]
    return float(lam / (len(I) * len(J)) - (1.0 / np.sqrt(len(I)) + 1.0 / np.sqrt(len(J))) ** 2)


def selection_null(n, m, ks, ls, rng, restarts=24, iters=8):
    """Search-adjusted null level of the rank-one energy share.  For every block size (k, l) a pure-noise
    n x m matrix is searched by alternating top-k / top-l selection (the same move family the swarm uses),
    and the pct-th percentile of the attained share lambda_1/(k l) over restarts is stored.  Subtracting this
    level removes the winner's-curse inflation that any optimiser creates on noise."""
    mu = np.zeros((len(ks), len(ls))); sd = np.zeros((len(ks), len(ls)))
    for a, k in enumerate(ks):
        for b, l in enumerate(ls):
            vals = []
            for r in range(restarts):
                Zn = standardize(rng.standard_normal((n, m)))
                J = rng.choice(m, l, replace=False)
                v = np.ones(l) / np.sqrt(l)
                for _ in range(iters):
                    I = np.argsort(-np.abs(Zn[:, J] @ v))[:k]
                    U, sv, Vt = np.linalg.svd(Zn[np.ix_(I, J)], full_matrices=False)
                    u = U[:, 0]
                    J = np.argsort(-np.abs(u @ Zn[I]))[:l]
                    v = np.linalg.svd(Zn[np.ix_(I, J)], full_matrices=False)[2][0]
                lam = np.linalg.svd(Zn[np.ix_(I, J)], compute_uv=False)[0] ** 2
                vals.append(lam / (k * l))
            mu[a, b] = np.mean(vals); sd[a, b] = np.std(vals, ddof=1)
    return mu, sd


class NullTable:
    def __init__(self, n, m, rng, ks=(10, 15, 25, 40, 60, 90, 130, 180, 250), ls=None):
        ls = ls or tuple(sorted(set([4, 5, 6, 8, 10, 13, 17, 22, max(23, m // 2)])))
        self.ks, self.ls = np.array(ks, float), np.array([x for x in ls if x <= m], float)
        self.mu, self.sd = selection_null(n, m, [int(k) for k in self.ks], [int(l) for l in self.ls], rng)
        from scipy.interpolate import RegularGridInterpolator
        g = (np.log(self.ks), np.log(self.ls))
        self.fm = RegularGridInterpolator(g, self.mu, bounds_error=False, fill_value=None)
        self.fs = RegularGridInterpolator(g, np.log(self.sd), bounds_error=False, fill_value=None)

    def __call__(self, k, l):
        k = np.clip(k, self.ks[0], self.ks[-1]); l = np.clip(l, self.ls[0], self.ls[-1])
        x = [[np.log(k), np.log(l)]]
        return float(self.fm(x)[0]), float(np.exp(self.fs(x)[0]))


def adjusted_score(Z, I, J, null):
    """Size-calibrated, search-adjusted coherence z-score used as the swarm objective:
    (lambda_1/(|I||J|) - mu0(|I|,|J|)) / sd0(|I|,|J|), with mu0 and sd0 the mean and standard deviation of
    the energy share that the same alternating search attains on pure noise of the same dimensions."""
    if len(I) < 3 or len(J) < 3:
        return -10.0
    S = Z[np.ix_(I, J)]
    G = S.T @ S if len(J) <= len(I) else S @ S.T
    lam = np.linalg.eigvalsh(G)[-1]
    mu, sd = null(len(I), len(J))
    return float((lam / (len(I) * len(J)) - mu) / sd)



def overlap_cells(a, b):
    """Overlap coefficient of the cell sets: shared cells divided by the smaller bicluster area."""
    I1, J1 = a; I2, J2 = b
    inter = len(np.intersect1d(I1, I2)) * len(np.intersect1d(J1, J2))
    return inter / min(len(I1) * len(J1), len(I2) * len(J2))


def jaccard_cells(a, b):
    I1, J1 = a; I2, J2 = b
    ni = len(np.intersect1d(I1, I2)); nj = len(np.intersect1d(J1, J2))
    inter = ni * nj
    union = len(I1) * len(J1) + len(I2) * len(J2) - inter
    return inter / union if union > 0 else 0.0


# ----------------------------------------------------------------------------------------------
# 3. LogitBoost (Friedman, Hastie, Tibshirani 2000) with depth-limited regression trees
# ----------------------------------------------------------------------------------------------
def _best_split(Fs, O, w, wz, node):
    """Best weighted least-squares split of the samples in `node` (boolean mask).
    Fs: features sorted per column, O: argsort per column.  Returns (gain, feature, threshold)."""
    ww = np.where(node, w, 0.0)[O]; zz = np.where(node, wz, 0.0)[O]
    WL = np.cumsum(ww, 0); SL = np.cumsum(zz, 0)
    W, S = WL[-1], SL[-1]
    WR = W[None, :] - WL; SR = S[None, :] - SL
    valid = (WL > 1e-12) & (WR > 1e-12)
    valid[:-1] &= Fs[1:] > Fs[:-1]
    valid[-1] = False
    gain = np.where(valid, SL ** 2 / np.maximum(WL, 1e-12) + SR ** 2 / np.maximum(WR, 1e-12), -np.inf)
    idx = np.argmax(gain); i, j = np.unravel_index(idx, gain.shape)
    if not np.isfinite(gain[i, j]):
        return None
    return gain[i, j], j, 0.5 * (Fs[i, j] + Fs[i + 1, j])


class _Tree2:
    """Depth-limited (<=2) weighted regression tree on presorted features (fast, deterministic)."""
    def fit(self, F, Fs, O, z, w, depth):
        n = len(z); wz = w * z
        self.root = self._grow(F, Fs, O, z, w, wz, np.ones(n, bool), depth)
        return self

    def _leaf(self, w, wz, node):
        W = w[node].sum()
        return ('leaf', wz[node].sum() / W if W > 0 else 0.0)

    def _grow(self, F, Fs, O, z, w, wz, node, depth):
        if depth == 0 or node.sum() < 4:
            return self._leaf(w, wz, node)
        sp = _best_split(Fs, O, w, wz, node)
        if sp is None:
            return self._leaf(w, wz, node)
        _, j, thr = sp
        left = node & (F[:, j] <= thr); right = node & ~left
        return ('split', j, thr, self._grow(F, Fs, O, z, w, wz, left, depth - 1), self._grow(F, Fs, O, z, w, wz, right, depth - 1))

    def predict(self, F):
        return self._pred(self.root, F)

    def _pred(self, nd, F):
        if nd[0] == 'leaf':
            return np.full(F.shape[0], nd[1])
        _, j, thr, l, r = nd
        out = np.empty(F.shape[0]); m = F[:, j] <= thr
        out[m] = self._pred(l, F[m]); out[~m] = self._pred(r, F[~m])
        return out


class LogitBoost:
    """Two-class LogitBoost (Friedman, Hastie and Tibshirani, 2000): Newton steps on the binomial
    log-likelihood, each fitting a depth-limited regression tree to the working response
    z = (y - p) / (p (1 - p)) with weights w = p (1 - p); F <- F + f/2 and p = 1 / (1 + exp(-2F))."""
    def __init__(self, n_rounds=50, depth=2, nu=1.0, zmax=4.0, seed=0):
        self.n_rounds, self.depth, self.nu, self.zmax, self.seed = n_rounds, depth, nu, zmax, seed

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y, float)
        O = np.argsort(X, axis=0, kind='stable'); Fs = np.take_along_axis(X, O, 0)
        F = np.zeros(len(y)); p = np.full(len(y), 0.5); self.trees = []
        for m in range(self.n_rounds):
            w = np.clip(p * (1 - p), 1e-6, None)
            z = np.clip((y - p) / w, -self.zmax, self.zmax)
            t = _Tree2().fit(X, Fs, O, z, w, self.depth)
            F += 0.5 * self.nu * t.predict(X)
            p = expit(2 * F); self.trees.append(t)
        return self

    def decision(self, X):
        X = np.asarray(X, float); F = np.zeros(X.shape[0])
        for t in self.trees:
            F += 0.5 * self.nu * t.predict(X)
        return F

    def predict_proba(self, X):
        return expit(2 * self.decision(X))


# ----------------------------------------------------------------------------------------------
# Membership features (all computed on the row-standardised discovery matrix Z)
# ----------------------------------------------------------------------------------------------
def _gene_features(Z, J, ref_genes):
    """Features of every gene relative to the shared condition profile v estimated from ref_genes over J."""
    S = Z[ref_genes][:, J]
    v = np.linalg.svd(S, full_matrices=False)[2][0]
    if v.sum() < 0:
        v = -v
    proj = Z[:, J] @ v / np.sqrt(len(J))             # signed projection on the profile
    sgn = np.sign(proj); sgn[sgn == 0] = 1
    notJ = np.setdiff1d(np.arange(Z.shape[1]), J)
    contrast = Z[:, J].mean(1) - Z[:, notJ].mean(1)
    zj = Z[:, J]
    cj = zj - zj.mean(1, keepdims=True)
    r = (cj @ (v - v.mean())) / (np.linalg.norm(cj, axis=1) * np.linalg.norm(v - v.mean()) + 1e-9)
    aligned = (zj * sgn[:, None])[:, np.argsort(-v)]
    return np.column_stack([proj, np.abs(proj), contrast * sgn, r * sgn, aligned])


def _cond_features(Z, I, J):
    S = Z[np.ix_(I, J)]
    u = np.linalg.svd(S, full_matrices=False)[0][:, 0]
    score = u @ Z[I] / np.sqrt(len(I))               # projection of every condition on the gene loading
    top = np.argsort(-np.abs(u))[:min(15, len(I))]
    aligned = (Z[I][top] * np.sign(u[top])[:, None]).T
    return np.column_stack([score, np.abs(score), aligned])


def _cap(p, keep_min, keep_max, tau):
    sel = np.where(p >= tau)[0]
    if len(sel) < keep_min:
        sel = np.argsort(-p)[:keep_min]
    if len(sel) > keep_max:
        sel = np.argsort(-p)[:keep_max]
    return np.sort(sel)


def crossfit_membership(Z, I, J, cfg, rng):
    """Cross-fitted LogitBoost update of gene and condition membership.  Labels come from the current
    bicluster; every probability is predicted out-of-fold, and the condition profile used to build a
    held-out gene's features is estimated without that gene, so no unit is scored by a model or a
    profile that already contains its own label."""
    n, m = Z.shape
    K = cfg['cf_folds']
    y = np.zeros(n, int); y[I] = 1
    folds = rng.permutation(n) % K
    p = np.zeros(n)
    for k in range(K):
        tr = folds != k; te = ~tr
        ref = np.where(tr & (y == 1))[0]
        if len(ref) < 3:
            return I, J
        F = _gene_features(Z, J, ref)
        lb = LogitBoost(cfg['lb_rounds'], cfg['lb_depth'], seed=int(rng.integers(1 << 30)))
        p[te] = lb.fit(F[tr], y[tr]).predict_proba(F[te])
    I_new = _cap(p, cfg['gmin'], cfg['gmax'], cfg['tau_g'])
    yc = np.zeros(m, int); yc[J] = 1
    Fc = _cond_features(Z, I_new, J)
    Kc = min(K, m); folds = rng.permutation(m) % Kc
    pc = np.zeros(m)
    for k in range(Kc):
        tr = folds != k; te = ~tr
        if yc[tr].sum() < 2:
            pc[te] = yc[te]; continue
        lb = LogitBoost(cfg['lb_rounds'], cfg['lb_depth'], seed=int(rng.integers(1 << 30)))
        pc[te] = lb.fit(Fc[tr], yc[tr]).predict_proba(Fc[te])
    J_new = _cap(pc, cfg['smin'], cfg['smax'], cfg['tau_c'])
    return I_new, J_new


def insample_membership(Z, I, J, cfg, rng):
    """Ablation: LogitBoost trained and scored on the same labels (the circular update of LB-BSO)."""
    n, m = Z.shape
    y = np.zeros(n, int); y[I] = 1
    F = _gene_features(Z, J, I)
    p = LogitBoost(cfg['lb_rounds'], cfg['lb_depth'], seed=int(rng.integers(1 << 30))).fit(F, y).predict_proba(F)
    I_new = _cap(p, cfg['gmin'], cfg['gmax'], cfg['tau_g'])
    yc = np.zeros(m, int); yc[J] = 1
    Fc = _cond_features(Z, I_new, J)
    pc = LogitBoost(cfg['lb_rounds'], cfg['lb_depth'], seed=int(rng.integers(1 << 30))).fit(Fc, yc).predict_proba(Fc)
    return I_new, _cap(pc, cfg['smin'], cfg['smax'], cfg['tau_c'])


# ----------------------------------------------------------------------------------------------
# Diversity-constrained binary bee swarm
# ----------------------------------------------------------------------------------------------
DEFAULT = dict(n_bees=20, t_max=40, stall=6, hv_tol=1e-3, limit=6, archive=20, lam=1.0, jac_admit=0.5,
               gmin=10, gmax=250, smin=4, smax=None, beta0=2.0, eta=1.5, temp=2.0, n_random=40,
               lb_rounds=50, lb_depth=2, cf_folds=5, tau_g=0.4, tau_c=0.7, refine_rounds=2,
               eps=0.5, n_perm=499, q=0.10, use_thinning=True, use_boost=True, use_crossfit=True,
               use_diversity=True, use_selnull=True, n_refine=8, anchor_temp=1.0, polish=1, adaptive=False, member_cert=False, objective='delta')


def pearson_score(Z, I, J):
    """Base-paper style objective: share of gene pairs with |Pearson r| >= 0.7 over J."""
    S = _stdrows(Z[np.ix_(I, J)]); R = np.abs(S @ S.T) / len(J)
    iu = np.triu_indices(len(I), 1)
    return float(np.mean(R[iu] >= 0.7)) - 0.5


class Swarm:
    def __init__(self, Z, cfg, rng, null=None):
        self.Z, self.cfg, self.rng, self.null = Z, cfg, rng, null
        self.n, self.m = Z.shape
        if cfg['smax'] is None:
            cfg['smax'] = max(cfg['smin'] + 1, self.m // 2)
        self.n_eval = 0
        self.R = None
        self.tabu = None

    def score(self, I, J):
        self.n_eval += 1
        if self.cfg['objective'] == 'pearson':
            return pearson_score(self.Z, I, J)
        if self.null is not None:
            return adjusted_score(self.Z, I, J, self.null)
        return coherence(self.Z, I, J)

    def obj(self, I, J):
        d = self.score(I, J)
        return d if self.null is not None else d * len(I) * len(J)

    def fitness(self, I, J, archive, skip=None):
        d = self.score(I, J)
        f = d if self.null is not None else (d * len(I) * len(J) if d > 0 else d)
        if self.cfg['use_diversity'] and f > 0:
            js = [overlap_cells((I, J), (a['I'], a['J'])) for a in archive if a is not skip]
            if js:
                f *= (1.0 - self.cfg['lam'] * min(1.0, max(js)))
        return f, d

    def seed(self):
        """Seed a food source on the discovery fold.  Two seed types are alternated:
        (a) anchor gene -> its most correlated partners (|r|) -> conditions on which the group deviates;
        (b) anchor gene -> the conditions where it deviates most -> genes co-deviating on them."""
        c = self.cfg
        if self.R is None:
            self.R = (self.Z @ self.Z.T) / self.m
            A = np.abs(self.R); np.fill_diagonal(A, 0)
            kc = np.sort(A, 1)[:, -20:].mean(1)              # local co-expression connectivity
            zc = (kc - kc.mean()) / (kc.std() + 1e-9)
            w = np.exp(c['anchor_temp'] * zc); self.anchor_p = w / w.sum()
        p = self.anchor_p.copy()
        if self.tabu is not None and len(self.tabu):
            p[self.tabu] *= 0.02                        # anchors already covered by the archive are rarely reused
        g0 = int(self.rng.choice(self.n, p=p / p.sum()))
        k = int(self.rng.integers(20, 61))
        l = int(self.rng.integers(c['smin'], min(c['smax'], c['smin'] + 6) + 1))
        if self.rng.random() < 0.5:
            r = self.R[g0]
            I = np.argsort(-np.abs(r))[:k]
            sgn = np.sign(r[I]); sgn[sgn == 0] = 1
            J = np.argsort(-np.abs(sgn @ self.Z[I]))[:l]
        else:
            J = np.argsort(-np.abs(self.Z[g0]))[:l]
            sc = self.Z[:, J] @ np.sign(self.Z[g0, J])
            I = np.argsort(-np.abs(sc))[:k]
        return np.sort(I), np.sort(J)

    def _bits(self, xi, xk, xg, guide):
        c = self.cfg; size = len(xi)
        phi = self.rng.uniform(-1, 1, size); r = self.rng.uniform(0, 1, size)
        v = c['beta0'] * (2 * xi - 1) + phi * (xi - xk) + r * (xg - xi) + c['eta'] * guide
        return self.rng.random(size) < expit(c['temp'] * v)

    def neighbour(self, src, other, leader):
        c = self.cfg; Z = self.Z
        I, J = src['I'], src['J']
        S = Z[np.ix_(I, J)]
        U, sv, Vt = np.linalg.svd(S, full_matrices=False)
        u, v = U[:, 0], Vt[0]
        pool = np.unique(np.concatenate([I, other['I'], leader['I'], self.rng.choice(self.n, c['n_random'], replace=False)]))
        g = np.abs(Z[np.ix_(pool, J)] @ v) / np.sqrt(len(J))
        g = (g - g.mean()) / (g.std() + 1e-9)
        xi = np.isin(pool, I).astype(float); xk = np.isin(pool, other['I']).astype(float); xg = np.isin(pool, leader['I']).astype(float)
        newI = pool[self._bits(xi, xk, xg, g)]
        allj = np.arange(self.m)
        sc = np.abs(u @ Z[I]) / np.sqrt(len(I))
        sc = (sc - sc.mean()) / (sc.std() + 1e-9)
        xi = np.isin(allj, J).astype(float); xk = np.isin(allj, other['J']).astype(float); xg = np.isin(allj, leader['J']).astype(float)
        newJ = allj[self._bits(xi, xk, xg, sc)]
        if len(newI) < c['gmin'] or len(newJ) < c['smin']:
            return None
        if len(newI) > c['gmax']:
            newI = newI[np.argsort(-np.abs(Z[np.ix_(newI, J)] @ v))[:c['gmax']]]
        if len(newJ) > c['smax']:
            newJ = newJ[np.argsort(-sc[newJ])[:c['smax']]]
        if c['polish'] > 0:
            newI, newJ = self.polish(newI, newJ, c['polish'])
        return np.sort(newI), np.sort(newJ)

    def polish(self, I, J, steps):
        """Coherence-guided local polish (memetic step).  Genes and conditions are ranked by their absolute
        projections on the current singular pair, and the block size along each axis is chosen to maximise
        the total search-adjusted excess energy |I||J| x score(I, J)."""
        Z, c = self.Z, self.cfg
        kgrid = np.unique(np.geomspace(c['gmin'], c['gmax'], 14).astype(int))
        for _ in range(steps):
            v = np.linalg.svd(Z[np.ix_(I, J)], full_matrices=False)[2][0]
            og = np.argsort(-np.abs(Z[:, J] @ v))
            ks = kgrid if c['adaptive'] else [len(I)]
            I = og[:max(ks, key=lambda k: self.obj(og[:k], J))]
            u = np.linalg.svd(Z[np.ix_(I, J)], full_matrices=False)[0][:, 0]
            oc = np.argsort(-np.abs(u @ Z[I]))
            ls = range(c['smin'], c['smax'] + 1) if c['adaptive'] else [len(J)]
            J = oc[:max(ls, key=lambda l: self.obj(I, oc[:l]))]
        return np.sort(I), np.sort(J)

    def hypervolume(self, archive):
        pts = sorted([(a['delta'], np.log(len(a['I']) * len(a['J'])) / np.log(self.n * self.m)) for a in archive if a['delta'] > 0], key=lambda t: -t[0])
        hv, best = 0.0, 0.0
        for x, y in pts:                              # 2-D dominated area w.r.t. reference (0, 0)
            if y > best:
                hv += x * (y - best); best = y
        return hv

    def update_archive(self, archive, I, J, d):
        if d <= 0:
            return archive
        cand = dict(I=I, J=J, delta=d, f=d if self.null is not None else d * len(I) * len(J))
        for k, a in enumerate(archive):
            if redundant(cand, a, self.cfg['jac_admit']):
                if cand['f'] > a['f']:
                    archive[k] = cand
                return archive
        archive.append(cand)
        archive.sort(key=lambda a: -a['f'])
        return archive[:self.cfg['archive']]

    def run(self):
        c = self.cfg
        srcs, archive = [], []
        for _ in range(c['n_bees']):
            I, J = self.seed(); f, d = self.fitness(I, J, archive)
            srcs.append(dict(I=I, J=J, f=f, d=d, trial=0))
            archive = self.update_archive(archive, I, J, d)
        hv_hist = [self.hypervolume(archive)]
        stall = 0
        for t in range(c['t_max']):
            fits = np.array([s['f'] for s in srcs])
            prob = np.exp((fits - fits.max()) / (np.std(fits) + 1e-9)); prob /= prob.sum()
            order = list(range(c['n_bees'])) + list(self.rng.choice(c['n_bees'], c['n_bees'], p=prob))  # employed + onlookers
            for i in order:
                k = int(self.rng.integers(c['n_bees'] - 1)); k += k >= i
                leader = archive[int(self.rng.integers(min(5, len(archive))))] if archive else srcs[i]
                nb = self.neighbour(srcs[i], srcs[k], leader)
                if nb is None:
                    srcs[i]['trial'] += 1; continue
                own = [a for a in archive if jaccard_cells((a['I'], a['J']), (srcs[i]['I'], srcs[i]['J'])) >= c['jac_admit']]
                f, d = self.fitness(nb[0], nb[1], [a for a in archive if not any(a is o for o in own)])
                if f > srcs[i]['f']:
                    srcs[i] = dict(I=nb[0], J=nb[1], f=f, d=d, trial=0)
                    archive = self.update_archive(archive, nb[0], nb[1], d)
                else:
                    srcs[i]['trial'] += 1
            for i in range(c['n_bees']):                   # scouts
                if srcs[i]['trial'] > c['limit']:
                    I, J = self.seed(); f, d = self.fitness(I, J, archive)
                    srcs[i] = dict(I=I, J=J, f=f, d=d, trial=0)
                    archive = self.update_archive(archive, I, J, d)
            self.tabu = np.unique(np.concatenate([a['I'] for a in archive])) if archive else None
            hv = self.hypervolume(archive); hv_hist.append(hv)
            rel = (hv - hv_hist[-2]) / (abs(hv_hist[-2]) + 1e-12)
            stall = stall + 1 if rel < c['hv_tol'] else 0
            if stall >= c['stall']:
                break
        return archive, hv_hist


# ----------------------------------------------------------------------------------------------
# Certification on the confirmation fold
# ----------------------------------------------------------------------------------------------
def certify(Z2, cands, cfg, rng, mean_expr):
    n, m = Z2.shape
    edges = np.quantile(mean_expr, np.linspace(0, 1, 11)[1:-1])
    bins = np.digitize(mean_expr, edges)
    out = []
    B = cfg['n_perm']
    for c in cands:
        I, J = np.asarray(c['I']), np.asarray(c['J'])
        T = coherence(Z2, I, J)
        null_col = np.array([coherence(Z2, I, rng.choice(m, len(J), replace=False)) for _ in range(B)])
        bI = bins[I]; ub, cnt = np.unique(bI, return_counts=True)
        pools = {b: np.where(bins == b)[0] for b in ub}
        null_row = np.empty(B)
        for r in range(B):
            Ir = np.concatenate([rng.choice(pools[b], k, replace=False) for b, k in zip(ub, cnt)])
            null_row[r] = coherence(Z2, Ir, J)
        p_col = (1 + np.sum(null_col >= T)) / (B + 1)
        p_row = (1 + np.sum(null_row >= T)) / (B + 1)
        out.append(dict(I=I, J=J, T=T, p_col=p_col, p_row=p_row, p=max(p_col, p_row), delta_disc=c.get('delta', np.nan), I0=c.get('I0', I), J0=c.get('J0', J)))
    return out


def certify_members(Z1, Z2, I, J, q, rng, B=999):
    """Member-level certification inside a certified bicluster, on the confirmation fold.
    Gene g: projection of its confirmation-fold profile over J on the leave-one-out bicluster profile
    (signs fixed by the discovery-fold loadings), tested against permutations of the gene's own row.
    Condition j: sign-aligned gene average at j, tested against the conditions outside J.
    Benjamini-Hochberg is applied within the bicluster for genes and for conditions separately."""
    I = np.asarray(I); J = np.asarray(J); m = Z2.shape[1]
    u1 = np.linalg.svd(Z1[np.ix_(I, J)], full_matrices=False)[0][:, 0]
    sg = np.sign(u1); sg[sg == 0] = 1
    if np.sum(sg * u1) < 0:
        sg = -sg
    A = Z2[I] * sg[:, None]                               # sign-aligned confirmation rows
    tot = A[:, J].sum(0)
    perms = np.argsort(rng.random((B, m)), 1)[:, :len(J)]  # random condition sets for the row null
    pg = np.empty(len(I))
    for a in range(len(I)):
        prof = (tot - A[a, J]) / max(len(I) - 1, 1)
        prof = prof / (np.linalg.norm(prof) + 1e-12)
        t = A[a, J] @ prof
        tn = A[a][perms] @ prof
        pg[a] = (1 + np.sum(tn >= t)) / (B + 1)
    keep_g, _ = bh(pg, q)
    col = A.mean(0)
    notJ = np.setdiff1d(np.arange(m), J)
    pc = np.array([(1 + np.sum(col[notJ] >= col[j])) / (1 + len(notJ)) for j in J])
    keep_c, _ = bh(pc, q)
    return I[keep_g], J[keep_c], pg, pc


def bh(p, q):
    p = np.asarray(p, float); n = len(p)
    if n == 0:
        return np.zeros(0, bool), np.zeros(0)
    o = np.argsort(p); ranked = p[o] * n / (np.arange(n) + 1)
    qv = np.minimum.accumulate(ranked[::-1])[::-1]
    qvals = np.empty(n); qvals[o] = np.minimum(qv, 1)
    return qvals <= q, qvals


def redundant(a, b, jac):
    """Two biclusters are redundant when their cells overlap (Jaccard >= jac) or when they share most genes
    on overlapping conditions (gene overlap coefficient >= 0.3 and condition overlap coefficient >= 0.5)."""
    if jaccard_cells((a['I'], a['J']), (b['I'], b['J'])) >= jac:
        return True
    oi = len(np.intersect1d(a['I'], b['I'])) / min(len(a['I']), len(b['I']))
    oj = len(np.intersect1d(a['J'], b['J'])) / min(len(a['J']), len(b['J']))
    return oi >= 0.3 and oj >= 0.5


def consolidate(cands, sw, cfg):
    """Merge complementary pieces of one module and drop redundant ones (discovery fold only).
    A candidate whose conditions largely coincide with a kept bicluster is merged into it when the union of
    their genes raises the total search-adjusted excess energy; otherwise it is dropped if redundant."""
    tot = sw.obj
    kept = []
    for c in sorted(cands, key=lambda c: -sw.obj(c['I'], c['J'])):
        merged = False
        for k, u in enumerate(kept):
            oj = len(np.intersect1d(c['J'], u['J'])) / min(len(c['J']), len(u['J']))
            if oj >= 0.5:
                Iu = np.union1d(u['I'], c['I'])
                if len(Iu) <= cfg['gmax'] and tot(Iu, u['J']) > tot(u['I'], u['J']):
                    kept[k] = dict(u, I=Iu, delta=sw.score(Iu, u['J'])); merged = True; break
            if redundant(c, u, cfg['jac_admit']):
                merged = True; break
        if not merged:
            kept.append(c)
    return kept


def thin_folds(X, lib, groups, eps, rng):
    phi = estimate_dispersion(X, lib, np.asarray(groups))
    X1, X2 = nb_thin(X, phi, eps, rng)
    return log_cpm(X1, lib * eps), log_cpm(X2, lib * (1 - eps)), phi


def run_hive(X, lib, groups, seed=0, folds=None, **over):
    """folds: optional precomputed (Y1, Y2) so that HIVE-Bic and baselines share one thinning draw."""
    cfg = dict(DEFAULT); cfg.update(over)
    rng = np.random.default_rng(seed)
    X = np.asarray(X); lib = np.asarray(lib, float)
    if folds is not None:
        Y1, Y2 = folds
    elif cfg['use_thinning']:
        Y1, Y2, _ = thin_folds(X, lib, groups, cfg['eps'], rng)
    else:                                               # double dipping: same data for search and test
        Y1 = Y2 = log_cpm(X, lib)
    Z1, Z2 = standardize(Y1), standardize(Y2)
    smax = cfg['smax'] if cfg['smax'] is not None else max(cfg['smin'] + 1, Z1.shape[1] // 2)
    null = NullTable(Z1.shape[0], Z1.shape[1], np.random.default_rng(12345)) if cfg['use_selnull'] else None
    sw = Swarm(Z1, cfg, rng, null)
    archive, hv = sw.run()
    cands = []
    for a in archive[:cfg['n_refine']]:
        I, J = a['I'], a['J']; I0, J0 = I, J
        if cfg['use_boost']:
            for _ in range(cfg['refine_rounds']):
                I2, J2 = (crossfit_membership if cfg['use_crossfit'] else insample_membership)(Z1, I, J, cfg, rng)
                same = len(I2) == len(I) and len(J2) == len(J) and np.all(I2 == I) and np.all(J2 == J)
                if sw.score(I2, J2) > 0:
                    I, J = I2, J2
                if same:
                    break
        d = sw.score(I, J)
        if d > 0:
            cands.append(dict(I=I, J=J, delta=d, I0=I0, J0=J0))
    uniq = consolidate(cands, sw, cfg)
    cert = certify(Z2, uniq, cfg, rng, Y2.mean(1))
    rej, qv = bh([c['p'] for c in cert], cfg['q'])
    for c, r, qq in zip(cert, rej, qv):
        c['certified'] = bool(r); c['q'] = float(qq)
        if r and cfg['member_cert']:
            Ic, Jc, pg, pc = certify_members(Z1, Z2, c['I'], c['J'], cfg['q'], rng)
            c['I_cert'], c['J_cert'] = Ic, Jc
            if len(Ic) < 3 or len(Jc) < 2:
                c['certified'] = False
        else:
            c['I_cert'], c['J_cert'] = c['I'], c['J']
    return dict(candidates=cert, hv=hv, Y1=Y1, Y2=Y2, cfg=cfg, n_eval=sw.n_eval, null=null)


In [ ]:
%%writefile synth.py
"""Semi-synthetic count matrices with planted biclusters, built from the empirical pig-atlas parameters."""
import numpy as np


def plant_effects(n, m, rng, delta, sizes_I=(30, 80), sizes_J=(5, 10)):
    """Three planted biclusters (shift, shift-scale, anti-correlated) on disjoint gene sets.
    Returns log2 effect matrix and ground-truth (I, J, type) list."""
    E = np.zeros((n, m))
    genes = rng.permutation(n)
    truth, start = [], 0
    for kind in ('shift', 'scale', 'anti'):
        k = int(rng.integers(sizes_I[0], sizes_I[1] + 1)); l = int(rng.integers(sizes_J[0], sizes_J[1] + 1))
        I = np.sort(genes[start:start + k]); start += k
        J = np.sort(rng.choice(m, l, replace=False))
        b = rng.uniform(0.5, 1.0, l)                            # shared condition profile
        if kind == 'shift':
            a = np.ones(k)
        elif kind == 'scale':
            a = rng.uniform(0.5, 1.5, k)
        else:
            a = np.where(rng.random(k) < 0.5, 1.0, -1.0)
        E[np.ix_(I, J)] += delta * np.outer(a, b)
        truth.append((I, J, kind))
    return E, truth


def make_matrix(base, phi, lib, rng, delta, background='nb', Xreal=None):
    """background='nb'  : NB counts with gene means / dispersions / library sizes of the real atlas (no structure)
       background='real': the real atlas counts themselves, with planted effects applied multiplicatively"""
    n, m = base.shape[0], len(lib)
    E, truth = plant_effects(n, m, rng, delta)
    fold = 2.0 ** E
    if background == 'nb':
        s = lib / np.exp(np.mean(np.log(lib)))
        mu = base[:, None] * s[None, :] * fold
        r = (1.0 / phi)[:, None]
        X = rng.negative_binomial(r * np.ones_like(mu), r / (r + mu))
    else:
        X = Xreal.copy().astype(np.int64)
        up = fold > 1; dn = fold < 1
        X[up] = X[up] + rng.poisson(X[up] * (fold[up] - 1.0))
        X[dn] = rng.binomial(X[dn], fold[dn])
    return X, truth


def match_scores(found, truth):
    """Prelic recovery / relevance with cell-level Jaccard."""
    from hive import jaccard_cells
    T = [(t[0], t[1]) for t in truth]
    if len(found) == 0:
        return 0.0, 0.0, np.zeros(len(T))
    S = np.array([[jaccard_cells(f, t) for t in T] for f in found])
    rec_each = S.max(0)
    return float(rec_each.mean()), float(S.max(1).mean()), rec_each


In [ ]:
%%writefile baselines.py
"""Nine biclustering baselines run on the discovery fold (log-CPM Y1 / row-standardised Z1).
Each returns a list of (I, J) index arrays (at most K biclusters)."""
import numpy as np
import warnings
from hive import standardize, coherence, jaccard_cells, LogitBoost, _stdrows

K = 10


def _dedupe(bics, Z, K=K, jac=0.5):
    bics = [(np.sort(np.asarray(I)), np.sort(np.asarray(J))) for I, J in bics if len(I) >= 3 and len(J) >= 3]
    bics.sort(key=lambda b: -coherence(Z, b[0], b[1]) * len(b[0]) * len(b[1]))
    out = []
    for b in bics:
        if all(jaccard_cells(b, o) < jac for o in out):
            out.append(b)
        if len(out) == K:
            break
    return out


def _biclustlib(algo, data):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        r = algo.run(np.asarray(data, float))
    return [(np.asarray(b.rows), np.asarray(b.cols)) for b in r.biclusters]


def cc(Y, Z, rng):
    from biclustlib.algorithms import ChengChurchAlgorithm
    return _dedupe(_biclustlib(ChengChurchAlgorithm(num_biclusters=K), Y), Z)


def las(Y, Z, rng):
    from biclustlib.algorithms import LargeAverageSubmatrices
    # LAS searches for large positive averages; run on Z and on -Z so that down-regulated blocks are reachable
    b = _biclustlib(LargeAverageSubmatrices(num_biclusters=K), Z) + _biclustlib(LargeAverageSubmatrices(num_biclusters=K), -Z)
    return _dedupe(b, Z)


def plaid(Y, Z, rng):
    from biclustlib.algorithms import Plaid
    return _dedupe(_biclustlib(Plaid(num_biclusters=K), Y), Z)


def isa(Y, Z, rng, n_seeds=200, tg=2.0, tc=2.0, iters=30):
    """Iterative signature algorithm (Bergmann et al.): alternate thresholded gene and condition scores."""
    Eg = _stdrows(Z.T).T                      # genes standardised across conditions (rows)
    Ec = _stdrows(Z)                          # conditions standardised across genes (columns)
    Ec = ((Z - Z.mean(0)) / (Z.std(0) + 1e-9))
    n, m = Z.shape; mods = []
    for _ in range(n_seeds):
        g = np.zeros(n); g[rng.choice(n, 20, replace=False)] = 1
        for _ in range(iters):
            sc = Ec.T @ g; sc = (sc - sc.mean()) / (sc.std() + 1e-9)
            c = np.where(np.abs(sc) > tc, sc, 0.0)
            if not c.any():
                break
            sg = Eg @ c; sg = (sg - sg.mean()) / (sg.std() + 1e-9)
            g_new = np.where(np.abs(sg) > tg, sg, 0.0)
            if np.allclose(np.sign(g_new), np.sign(g)):
                g = g_new; break
            g = g_new
            if not g.any():
                break
        I = np.where(g != 0)[0]; J = np.where(c != 0)[0] if g.any() else np.array([], int)
        if len(I) >= 3 and len(J) >= 3:
            mods.append((I, J))
    return _dedupe(mods, Z)


def ssvd(Y, Z, rng, layers=K):
    """Sparse SVD biclustering (Lee et al.): rank-one layers with soft-thresholded singular vectors,
    sparsity chosen by BIC on a grid, then deflation."""
    R = Z.copy(); out = []
    n, m = R.shape
    for _ in range(layers):
        U, s, Vt = np.linalg.svd(R, full_matrices=False)
        u, v = U[:, 0] * s[0], Vt[0]
        best = None
        for _ in range(10):
            # update u with soft threshold chosen by BIC
            cand = []
            z = R @ v
            for lam in np.quantile(np.abs(z), [0.5, 0.7, 0.8, 0.9, 0.95, 0.97, 0.98, 0.99]):
                uu = np.sign(z) * np.maximum(np.abs(z) - lam, 0)
                rss = np.sum((R - np.outer(uu, v)) ** 2); df = np.count_nonzero(uu)
                cand.append((n * m * np.log(rss / (n * m)) + np.log(n * m) * df, uu))
            u = min(cand, key=lambda t: t[0])[1]
            if not u.any():
                break
            cand = []
            z = R.T @ u
            for lam in np.quantile(np.abs(z), [0.3, 0.5, 0.7, 0.8, 0.9]):
                vv = np.sign(z) * np.maximum(np.abs(z) - lam, 0)
                rss = np.sum((R - np.outer(u, vv / (np.linalg.norm(vv) + 1e-12))) ** 2); df = np.count_nonzero(vv)
                cand.append((n * m * np.log(rss / (n * m)) + np.log(n * m) * df, vv))
            v = min(cand, key=lambda t: t[0])[1]
            nv = np.linalg.norm(v)
            if nv == 0:
                break
            v = v / nv
        I = np.where(u != 0)[0]; J = np.where(v != 0)[0]
        if len(I) < 3 or len(J) < 3:
            break
        out.append((I, J))
        R = R - np.outer(u, v)
    return _dedupe(out, Z)


def spectral_bic(Y, Z, rng):
    from sklearn.cluster import SpectralBiclustering
    P = 2.0 ** Y                              # strictly positive scale required
    mdl = SpectralBiclustering(n_clusters=(6, 5), method='log', random_state=int(rng.integers(1 << 30))).fit(P)
    bics = []
    for r in range(6):
        for c in range(5):
            I = np.where(mdl.row_labels_ == r)[0]; J = np.where(mdl.column_labels_ == c)[0]
            if 3 <= len(I) <= 400 and len(J) >= 3:
                bics.append((I, J))
    return _dedupe(bics, Z)


def spectral_cocl(Y, Z, rng):
    from sklearn.cluster import SpectralCoclustering
    P = 2.0 ** Y
    mdl = SpectralCoclustering(n_clusters=K, random_state=int(rng.integers(1 << 30))).fit(P)
    bics = [(np.where(mdl.row_labels_ == k)[0], np.where(mdl.column_labels_ == k)[0]) for k in range(K)]
    return _dedupe(bics, Z)


def _msr(Yb):
    return float(np.mean((Yb - Yb.mean(1, keepdims=True) - Yb.mean(0, keepdims=True) + Yb.mean()) ** 2))


def bpso(Y, Z, rng, n_part=20, iters=30, w=0.7, c1=1.5, c2=1.5):
    """Binary PSO biclustering with a Cheng-Church mean-squared-residue fitness (Kennedy-Eberhart transfer)."""
    n, m = Y.shape
    delta = 0.5 * _msr(Y)
    def fit(x):
        I = np.where(x[:n] > 0)[0]; J = np.where(x[n:] > 0)[0]
        if len(I) < 3 or len(J) < 3:
            return -1e9
        r = _msr(Y[np.ix_(I, J)])
        size = np.log(len(I) * len(J))
        return size if r <= delta else size * delta / r
    X = np.zeros((n_part, n + m))
    for p in range(n_part):
        X[p, rng.choice(n, 40, replace=False)] = 1; X[p, n + rng.choice(m, 8, replace=False)] = 1
    V = rng.normal(0, 1, X.shape); pb = X.copy(); pf = np.array([fit(x) for x in X]); g = pb[np.argmax(pf)].copy()
    for _ in range(iters):
        r1, r2 = rng.random(X.shape), rng.random(X.shape)
        V = w * V + c1 * r1 * (pb - X) + c2 * r2 * (g - X)
        V = np.clip(V, -4, 4)
        X = (rng.random(X.shape) < 1 / (1 + np.exp(-V))).astype(float)
        X[:, :n] *= (rng.random((n_part, n)) < 0.05) | (X[:, :n] * (pb[:, :n] + g[None, :n]) > 0)   # sparsity guard
        f = np.array([fit(x) for x in X])
        better = f > pf; pb[better] = X[better]; pf[better] = f[better]
        g = pb[np.argmax(pf)].copy()
    bics = [(np.where(x[:n] > 0)[0], np.where(x[n:] > 0)[0]) for x in pb]
    return _dedupe(bics, Z)


def lbbso(Y, Z, rng, n_bees=20, lb_iter=50, bso_iter=30, alpha=0.7, beta=1.5, gamma=1.5, thr=0.7):
    """Re-implementation of the LB-BSO procedure described in the base manuscript: random binary membership,
    in-sample LogitBoost relabelling of gene membership, PSO-type displacement on membership scores and a
    Pearson-correlation fitness with a 0.7 threshold.  Where the original description is under-specified,
    the most direct reading was implemented (see the paper text)."""
    n, m = Y.shape
    def fit(I, J):
        if len(I) < 3 or len(J) < 3:
            return -1.0
        S = _stdrows(Y[np.ix_(I, J)]); R = np.abs(S @ S.T) / len(J)
        iu = np.triu_indices(len(I), 1)
        return float(np.mean(R[iu] >= thr))
    pos = np.zeros((n_bees, n + m))
    for b in range(n_bees):
        pos[b, rng.choice(n, 40, replace=False)] = 1; pos[b, n + rng.choice(m, 8, replace=False)] = 1
    vel = np.zeros_like(pos)
    for b in range(n_bees):                    # LogitBoost relabelling (labels = current membership)
        I = np.where(pos[b, :n] > 0.5)[0]; J = np.where(pos[b, n:] > 0.5)[0]
        y = np.zeros(n, int); y[I] = 1
        p = LogitBoost(lb_iter, 2, seed=int(rng.integers(1 << 30))).fit(Y[:, J], y).predict_proba(Y[:, J])
        pos[b, :n] = (p >= 0.5).astype(float)
    pbest = pos.copy(); pf = np.array([fit(np.where(x[:n] > .5)[0], np.where(x[n:] > .5)[0]) for x in pos])
    g = pbest[np.argmax(pf)].copy()
    for _ in range(bso_iter):
        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        vel = alpha * vel + beta * r1 * (pbest - pos) + gamma * r2 * (g - pos)
        pos = np.clip(pos + vel, 0, 1)
        f = np.array([fit(np.where(x[:n] > .5)[0], np.where(x[n:] > .5)[0]) for x in pos])
        better = f > pf; pbest[better] = pos[better]; pf[better] = f[better]
        g = pbest[np.argmax(pf)].copy()
    bics = [(np.where(x[:n] > .5)[0], np.where(x[n:] > .5)[0]) for x in pbest]
    return _dedupe(bics, Z)


METHODS = dict(CC=cc, ISA=isa, Plaid=plaid, LAS=las, SSVD=ssvd, SpecBic=spectral_bic, SpecCo=spectral_cocl, BPSO=bpso, LBBSO=lbbso)


In [ ]:
%%writefile orthologs.py
"""One-to-one pig-human orthologs from NCBI gene_orthologs + gene_info (Ensembl IDs from dbXrefs)."""
import pandas as pd

def ens_of(dbx):
    for t in str(dbx).split('|'):
        if t.startswith('Ensembl:'):
            return t.split(':', 1)[1]
    return None

def build_one2one(orth, info):
    o = orth.copy()
    for c in ['#tax_id', 'GeneID', 'Other_tax_id', 'Other_GeneID']: o[c] = pd.to_numeric(o[c], errors='coerce')
    o = o[(o['#tax_id'] == 9606) & (o.Other_tax_id == 9823) & (o.relationship == 'Ortholog')]
    o = o.rename(columns={'GeneID': 'h', 'Other_GeneID': 'p'})[['h', 'p']].dropna().astype(int)
    o = o[~o.h.duplicated(keep=False) & ~o.p.duplicated(keep=False)]
    info = info.copy(); info['GeneID'] = pd.to_numeric(info.GeneID); info['tax_id'] = pd.to_numeric(info.tax_id)
    info['ens'] = info.dbXrefs.map(ens_of)
    hs = info[info.tax_id == 9606].set_index('GeneID'); ps = info[info.tax_id == 9823].set_index('GeneID')
    o = o[o.h.isin(hs.index) & o.p.isin(ps.index)]
    o = o.assign(h_ens=hs.ens.reindex(o.h).values, h_sym=hs.Symbol.reindex(o.h).values,
                 p_ens=ps.ens.reindex(o.p).values, p_sym=ps.Symbol.reindex(o.p).values)
    o = o.dropna(subset=['h_ens', 'p_ens'])
    o = o[~o.h_ens.duplicated(keep=False) & ~o.p_ens.duplicated(keep=False)]
    return o.sort_values('h').reset_index(drop=True)


In [ ]:
%%writefile make_annotation.py
"""Writes E-MTAB-5895_sample_annotation.tsv (run -> tissue, animal, age, breed, sex, stage), condensed from the ArrayExpress SDRF."""
import os
os.chdir(os.environ.get('HIVE_DATA', '.'))
T=["pituitary gland","cortex of kidney","heart left ventricle","omentum","mesenteric lymph node","spleen","tonsil","epididymis","penis","alveolus of lung","skeletal muscle tissue","caecum","uterus","colon","duodenum","ileum","stomach","hippocampal formation","amygdala","frontal cortex","occipital cortex","brainstem","medulla oblongata","pons","cerebellum","corpus callosum","dorsal plus ventral thalamus"]
I=["SUS_RI_Duroc_21","SUS_RI_Duroc_22","SUS_RI_Duroc_23","SUS_RI_Duroc_24"]
A=["4 month/Duroc/female/juvenile stage","4 month/Duroc/male/juvenile stage","5 month/Duroc/female/juvenile stage","3 month/Duroc/male/juvenile stage"]
P="""262,0,0,0;263,1,0,0;264,2,0,0;265,3,0,0;266,4,1,1;267,5,1,1;268,6,1,1;269,7,1,1;270,8,1,1;271,9,1,1;272,3,1,1;273,10,1,1;274,11,0,0;275,11,0,0;276,4,0,0;277,4,0,0;278,10,0,0;279,10,0,0;280,5,0,0;281,5,0,0;282,9,0,0;283,9,0,0;284,1,1,1;285,1,1,1;286,2,1,1;287,2,1,1;288,11,1,1;289,11,1,1;290,0,0,0;291,0,0,0;292,1,0,0;293,1,0,0;294,2,0,0;295,2,0,0;296,11,0,0;297,11,0,0;298,4,0,0;299,4,0,0;300,10,0,0;301,10,0,0;302,5,0,0;303,5,0,0;304,9,0,0;305,9,0,0;306,3,0,0;307,3,0,0;308,1,1,1;309,1,1,1;310,2,1,1;311,2,1,1;312,11,1,1;313,11,1,1;314,4,1,1;315,4,1,1;316,5,1,1;317,5,1,1;318,6,1,1;319,6,1,1;320,7,1,1;321,7,1,1;322,8,1,1;323,8,1,1;324,9,1,1;325,9,1,1;326,3,1,1;327,3,1,1;328,10,1,1;329,10,1,1;330,0,0,0;331,0,0,0;332,1,0,0;333,1,0,0;334,2,0,0;335,2,0,0;336,11,0,0;337,11,0,0;338,4,0,0;339,4,0,0;340,10,0,0;341,10,0,0;342,5,0,0;343,5,0,0;344,9,0,0;345,9,0,0;346,3,0,0;347,3,0,0;348,1,1,1;349,1,1,1;350,2,1,1;351,2,1,1;352,11,1,1;353,11,1,1;354,4,1,1;355,4,1,1;356,5,1,1;357,5,1,1;358,6,1,1;359,6,1,1;360,7,1,1;361,7,1,1;362,8,1,1;363,8,1,1;364,9,1,1;365,9,1,1;366,3,1,1;367,3,1,1;368,10,1,1;369,10,1,1;370,0,0,0;371,1,0,0;372,2,0,0;373,11,0,0;374,5,0,0;375,9,0,0;376,3,0,0;377,1,1,1;378,2,1,1;379,11,1,1;380,4,1,1;381,5,1,1;382,7,1,1;383,8,1,1;384,9,1,1;385,3,1,1;386,10,1,1;387,11,0,0;388,11,0,0;389,4,0,0;390,4,0,0;391,12,0,0;392,12,0,0;393,4,1,1;394,4,1,1;395,6,1,1;396,6,1,1;397,7,1,1;398,7,1,1;399,8,1,1;400,8,1,1;401,9,0,0;402,9,0,0;403,11,1,1;404,11,1,1;405,9,1,1;406,9,1,1;407,0,2,2;408,0,2,2;409,13,2,2;410,13,2,2;411,14,2,2;412,14,2,2;413,15,2,2;414,15,2,2;415,16,2,2;416,16,2,2;417,0,3,3;418,0,3,3;419,13,3,3;420,13,3,3;421,14,3,3;422,14,3,3;423,15,3,3;424,15,3,3;425,16,3,3;426,16,3,3;427,17,2,2;428,17,2,2;429,18,3,3;430,18,3,3;431,19,2,2;432,19,2,2;433,20,2,2;434,20,2,2;435,21,2,2;436,21,2,2;437,22,2,2;438,22,2,2;439,23,2,2;440,23,2,2;441,18,2,2;442,18,2,2;443,24,2,2;444,24,2,2;445,25,2,2;446,25,2,2;447,26,2,2;448,26,2,2;449,19,3,3;450,19,3,3;451,20,3,3;452,20,3,3;453,21,3,3;454,21,3,3;455,22,3,3;456,22,3,3;457,23,3,3;458,23,3,3;459,24,3,3;460,24,3,3;461,25,3,3;462,25,3,3;463,17,3,3;464,17,3,3;465,26,3,3;466,26,3,3;467,11,0,0;468,11,0,0;469,4,0,0;470,4,0,0;471,12,0,0;472,12,0,0;473,4,1,1;474,4,1,1;475,6,1,1;476,6,1,1;477,7,1,1;478,7,1,1;479,8,1,1;480,8,1,1;481,9,0,0;482,9,0,0;483,11,1,1;484,11,1,1;485,9,1,1;486,9,1,1;487,0,2,2;488,0,2,2;489,13,2,2;490,13,2,2;491,14,2,2;492,14,2,2;493,15,2,2;494,15,2,2;495,16,2,2;496,16,2,2;497,17,2,2;498,17,2,2;499,0,3,3;500,0,3,3;501,13,3,3;502,13,3,3;503,14,3,3;504,14,3,3;505,15,3,3;506,15,3,3;507,16,3,3;508,16,3,3;509,18,3,3;510,18,3,3;511,19,2,2;512,19,2,2;513,20,2,2;514,20,2,2;515,21,2,2;516,21,2,2;517,22,2,2;518,22,2,2;519,23,2,2;520,23,2,2;521,18,2,2;522,18,2,2;523,24,2,2;524,24,2,2;525,25,2,2;526,25,2,2;527,11,0,0;528,11,0,0;529,4,0,0;530,4,0,0;531,12,0,0;532,12,0,0;533,4,1,1;534,4,1,1;535,6,1,1;536,6,1,1;537,7,1,1;538,7,1,1;539,8,1,1;540,8,1,1;541,9,0,0;542,9,0,0;543,11,1,1;544,11,1,1;545,9,1,1;546,9,1,1;547,26,2,2;548,26,2,2;549,19,3,3;550,19,3,3;551,20,3,3;552,20,3,3;553,21,3,3;554,21,3,3;555,22,3,3;556,22,3,3;557,23,3,3;558,23,3,3;559,24,3,3;560,24,3,3;561,25,3,3;562,25,3,3;563,17,3,3;564,17,3,3;565,26,3,3;566,26,3,3"""
import csv
with open('E-MTAB-5895_sample_annotation.tsv','w') as f:
    f.write('run\ttissue\tindividual\tage\tbreed\tsex\tstage\n')
    for e in P.split(';'):
        r,t,i,a=map(int,e.split(','))
        age,breed,sex,st=A[a].split('/')
        f.write(f'ERR{1824000+r}\t{T[t]}\t{I[i]}\t{age}\t{breed}\t{sex}\t{st}\n')


In [ ]:
%%writefile download_data.py
"""Download every input of HIVE-Bic into $HIVE_DATA (default ./data).
Sources: Expression Atlas FTP (E-MTAB-5895 raw counts), Hugging Face mirrors of NCBI gene_orthologs / gene_info,
ToppGene GO biological process sets and GTEx v10 log2(TPM+1)."""
import os, sys, subprocess, urllib.request
import pandas as pd
from huggingface_hub import hf_hub_download
D = os.environ.get('HIVE_DATA', 'data'); os.makedirs(D, exist_ok=True); os.environ['HIVE_DATA'] = D
COUNTS = 'E-MTAB-5895-raw-counts.tsv.undecorated'
URLS = ['https://ftp.ebi.ac.uk/pub/databases/microarray/data/atlas/experiments/E-MTAB-5895/' + COUNTS,
        'http://ftp.ebi.ac.uk/pub/databases/microarray/data/atlas/experiments/E-MTAB-5895/' + COUNTS]

def get_counts():
    dst = f'{D}/{COUNTS}'
    if os.path.exists(dst): return
    for u in URLS:
        try:
            urllib.request.urlretrieve(u, dst); print('counts from', u); return
        except Exception as e:
            print('failed', u, e)
    sys.exit('Download the raw counts of E-MTAB-5895 from https://www.ebi.ac.uk/gxa/experiments/E-MTAB-5895/Downloads '
             f'and place {COUNTS} in {D}/, then re-run.')

def hf(repo, fn, out=None):
    p = hf_hub_download(repo, fn, repo_type='dataset')
    if out: os.replace(p, f'{D}/{out}') if not os.path.exists(f'{D}/{out}') else None
    return p

if __name__ == '__main__':
    get_counts()
    subprocess.run([sys.executable, os.path.join(os.path.dirname(os.path.abspath(__file__)), 'make_annotation.py')], check=True)
    if not os.path.exists(f'{D}/gene_orthologs.parquet'):
        pd.read_parquet(hf('pankajrajdeo/gene_orthologs', 'gene_orthologs.parquet')).to_parquet(f'{D}/gene_orthologs.parquet')
    if not os.path.exists(f'{D}/gene_info_human_pig.parquet'):
        parts = []
        for k in (27, 30):
            g = pd.read_parquet(hf('huggingworld/ncbi-all-data-gene-info', f'data/all-data-gene-info_part_{k:05d}.parquet'),
                                columns=['tax_id', 'GeneID', 'Symbol', 'dbXrefs', 'type_of_gene'])
            g['tax_id'] = pd.to_numeric(g.tax_id, errors='coerce'); parts.append(g[g.tax_id.isin([9606, 9823])])
        pd.concat(parts).to_parquet(f'{D}/gene_info_human_pig.parquet')
    if not os.path.exists(f'{D}/GeneOntologyBiologicalProcess(ToppGene).parquet'):
        pd.read_parquet(hf('pankajrajdeo/GeneOntologyBiologicalProcess_ToppGene', 'GeneOntologyBiologicalProcess(ToppGene).parquet')).to_parquet(f'{D}/GeneOntologyBiologicalProcess(ToppGene).parquet')
    print('downloads complete')


In [ ]:
%%writefile data_prep.py
"""Data preparation for HIVE-Bic: pig tissue atlas E-MTAB-5895 + NCBI one-to-one orthologs."""
import numpy as np, pandas as pd, os, json
D = os.environ.get('HIVE_DATA', 'data')

def load_pig(n_genes=2000, min_cpm=1.0, min_samples=3):
    raw = pd.read_csv(f'{D}/E-MTAB-5895-raw-counts.tsv.undecorated', sep='\t', index_col=0)
    ann = pd.read_csv(f'{D}/E-MTAB-5895_sample_annotation.tsv', sep='\t').set_index('run')
    ann = ann.loc[raw.columns]
    key = ann.tissue + '|' + ann.individual
    # sum sequencing runs of the same biological sample (tissue x animal)
    X = raw.T.groupby(key.values).sum().T
    meta = ann.assign(key=key.values).drop_duplicates('key').set_index('key').loc[X.columns, ['tissue','individual','sex','age']]
    if not os.path.exists(f'{D}/ncbi_pig_human_one2one.tsv'):
        from orthologs import build_one2one
        build_one2one(pd.read_parquet(f'{D}/gene_orthologs.parquet'), pd.read_parquet(f'{D}/gene_info_human_pig.parquet')).to_csv(f'{D}/ncbi_pig_human_one2one.tsv', sep='\t', index=False)
    orth = pd.read_csv(f'{D}/ncbi_pig_human_one2one.tsv', sep='\t')
    X = X.loc[X.index.intersection(orth.p_ens)]
    lib = raw.T.groupby(key.values).sum().T.sum(axis=0).loc[X.columns]  # full library size (all genes)
    cpm = X / lib * 1e6
    keep = (cpm >= min_cpm).sum(axis=1) >= min_samples
    X = X[keep]
    lc = np.log2(X / lib * 1e6 + 1)
    # rank genes by variance of log-CPM; keep top n_genes
    v = lc.var(axis=1).sort_values(ascending=False)
    X = X.loc[v.index[:n_genes]]
    o = orth.set_index('p_ens').loc[X.index]
    genes = pd.DataFrame({'pig_ens': X.index, 'pig_sym': o.p_sym.values, 'human_ens': o.h_ens.values,
                          'human_sym': o.h_sym.values, 'human_geneid': o.h.values})
    meta['lib_size'] = lib.loc[X.columns].values
    return X.astype(np.int64), meta, genes, int(keep.sum())

if __name__ == '__main__':
    X, meta, genes, n_expr = load_pig()
    print(X.shape, n_expr); print(meta.tissue.value_counts().head()); print(meta.lib_size.describe())
    X.to_csv(f'{D}/pig_counts_2000.tsv', sep='\t'); meta.to_csv(f'{D}/pig_samples.tsv', sep='\t'); genes.to_csv(f'{D}/pig_genes_2000.tsv', sep='\t', index=False)
    json.dump({'n_expressed_orthologs': n_expr}, open(f'{D}/prep_info.json','w'))


In [ ]:
%%writefile gtex_fetch.py
"""Fetch GTEx v10 log2(TPM+1) columns for the human orthologs of the pig gene universe (Hugging Face mirror)."""
import pandas as pd, pyarrow.parquet as pq, os, sys
from huggingface_hub import HfFileSystem
D = os.environ.get('HIVE_DATA', 'data')
REPO = 'datasets/silicobio/hoike_normal_expression_GTEx_Analysis_v10_log2tpmplus1/'
F = 'GTEx_Analysis_2022-06-06_v10_RNASeQCv2.4.2_gene_tpm_non_lcm_log2tpmplus1.parquet'
TISSUES = ['Brain','Heart','Muscle','Lung','Spleen','Kidney','Pituitary','Stomach','Colon','Small Intestine','Adipose Tissue','Uterus']
def main():
    genes = pd.read_csv(f'{D}/pig_genes_2000.tsv', sep='\t')
    fs = HfFileSystem(); pf = pq.ParquetFile(fs.open(REPO + F))
    cols = [c for c in genes.human_ens if c in set(pf.schema_arrow.names)]
    meta = pf.read(columns=['sample_id','tissue_type']).to_pandas()
    keep = meta.tissue_type.isin(TISSUES).values
    out = []
    for i in range(0, len(cols), 200):
        t = pf.read(columns=cols[i:i+200]).to_pandas()
        out.append(t[keep].astype('float32')); print(i, flush=True)
    E = pd.concat(out, axis=1); E.index = meta.sample_id[keep].values
    E.insert(0, 'tissue_type', meta.tissue_type[keep].values)
    E.to_parquet(f'{D}/gtex_v10_subset.parquet'); print(E.shape)
if __name__ == '__main__': main()


In [ ]:
%%writefile run_synth.py
"""Semi-synthetic benchmark: HIVE-Bic, its ablations and nine baselines on shared thinning folds."""
import numpy as np, pandas as pd, sys, json, time, os
from hive import *
from synth import *
from baselines import METHODS
D = os.environ.get('HIVE_DATA', 'data'); OUT = os.environ.get('HIVE_OUT', 'results')
X = pd.read_csv(f'{D}/pig_counts_2000.tsv', sep='\t', index_col=0); meta = pd.read_csv(f'{D}/pig_samples.tsv', sep='\t', index_col=0)
lib = meta.lib_size.values; groups = meta.tissue.values
phi = estimate_dispersion(X.values, lib, groups); base = (X.values / size_factors(lib)).mean(1)

def false_frac(found, truth):
    if not found: return np.nan
    f = 0
    for I, J in found:
        ov = max(overlap_cells((I, J), (t[0], t[1])) for t in truth) if truth else 0.0
        f += ov < 0.1
    return f / len(found)

def evaluate(name, found, cert_flags, truth, t_used, extra=None):
    rec, rel, each = match_scores(found, truth) if truth else (np.nan, np.nan, [])
    fc = [b for b, c in zip(found, cert_flags) if c]
    recc, relc, eachc = match_scores(fc, truth) if truth else (np.nan, np.nan, [])
    r = dict(method=name, n_out=len(found), n_cert=len(fc), rec=rec, rel=rel, rec_cert=recc, rel_cert=relc,
             false_out=false_frac(found, truth), false_cert=false_frac(fc, truth), time=t_used,
             any_false_cert=int(any(max([overlap_cells(b, (t[0], t[1])) for t in truth] or [0]) < 0.1 for b in fc)))
    if truth:
        for (I, J, kind), e, ec in zip(truth, each if len(each) else [0]*3, eachc if len(eachc) else [0]*3):
            r[f'rec_{kind}'] = float(e); r[f'recc_{kind}'] = float(ec)
    if extra: r.update(extra)
    return r

def run_one(scenario, background, delta, seed, variants, baselines=True):
    rng = np.random.default_rng(seed)
    if delta > 0:
        Xs, truth = make_matrix(base, phi, lib, rng, delta, background, Xreal=X.values)
    else:
        Xs, truth = make_matrix(base, phi, lib, rng, 0.0, background, Xreal=X.values); truth = []
    Y1, Y2, _ = thin_folds(Xs, lib, groups, 0.5, rng)
    Z1, Z2 = standardize(Y1), standardize(Y2)
    rows = []
    for vname, over in variants.items():
        t = time.time()
        if over.get('use_thinning', True) is False or 'eps' in over:
            res = run_hive(Xs, lib, groups, seed=seed, **over)
        else:
            res = run_hive(Xs, lib, groups, seed=seed, folds=(Y1, Y2), **over)
        tt = time.time() - t
        found = [(c['I'], c['J']) for c in res['candidates']]
        flags = [c['certified'] for c in res['candidates']]
        rows.append(evaluate(vname, found, flags, truth, tt, dict(n_iter=len(res['hv']) - 1, n_eval=res['n_eval'], hv=res['hv'])))
    if baselines:
        cfg = dict(DEFAULT); cfg['n_perm'] = 199
        for name, f in METHODS.items():
            t = time.time(); found = f(Y1, Z1, np.random.default_rng(seed)); tt = time.time() - t
            cert = certify(Z2, [dict(I=I, J=J) for I, J in found], cfg, np.random.default_rng(seed + 7), Y2.mean(1))
            rej, _ = bh([c['p'] for c in cert], cfg['q'])
            rows.append(evaluate(name, found, list(rej), truth, tt))
    for r in rows:
        r.update(scenario=scenario, background=background, delta=delta, seed=seed)
    return rows

if __name__ == '__main__':
    exp = sys.argv[1]; part = int(sys.argv[2]); nparts = int(sys.argv[3])
    jobs = []
    NS = int(os.environ.get('HIVE_SEEDS', 12))
    FULL = {'HIVE-Bic': {}}
    ABL = {'HIVE-Bic': {}, 'w/o thinning': dict(use_thinning=False), 'w/o search-adjusted null': dict(use_selnull=False),
           'w/o diversity': dict(use_diversity=False), 'w/o LogitBoost': dict(use_boost=False),
           'in-sample LogitBoost': dict(use_crossfit=False), 'w/o polish': dict(polish=0),
           'Pearson objective': dict(objective='pearson', use_selnull=False)}
    if exp == 'main':
        for d in (1.5, 2.0, 2.5):
            for s in range(NS): jobs.append(("main", "nb", d, 1000 + s, FULL, True))
        for s in range(NS): jobs.append(("main", "real", 2.0, 2000 + s, FULL, True))
    elif exp == 'null':
        for s in range(NS): jobs.append(("null", "nb", 0.0, 3000 + s, {'HIVE-Bic': {}, 'w/o thinning': dict(use_thinning=False)}, True))
    elif exp == 'ablation':
        for s in range(NS): jobs.append(("ablation", "nb", 2.0, 1000 + 30 + s, ABL, False))
    elif exp == 'sens':
        grid = {}
        for v in (10, 20, 40): grid[f'n_bees={v}'] = dict(n_bees=v)
        for v in (10, 25, 50, 100): grid[f'lb_rounds={v}'] = dict(lb_rounds=v)
        for v in (0.3, 0.5, 0.7): grid[f'eps={v}'] = dict(eps=v)
        for v in (0.0, 0.5, 1.0): grid[f'lam={v}'] = dict(lam=v)
        for v in (0.3, 0.4, 0.5): grid[f'tau_g={v}'] = dict(tau_g=v)
        for s in range(max(2, NS // 2)): jobs.append(('sens', 'nb', 2.0, 4000 + s, grid, False))
    jobs = jobs[part::nparts]
    fn = f'{OUT}/{exp}_part{part}.jsonl'
    done = set()
    if os.path.exists(fn):
        for l in open(fn):
            r = json.loads(l); done.add((r['scenario'], r['background'], r['delta'], r['seed']))
    for (sc, bg, d, s, var, bl) in jobs:
        if (sc, bg, d, s) in done: continue
        rows = run_one(sc, bg, d, s, var, bl)
        with open(fn, 'a') as f:
            for r in rows: f.write(json.dumps(r, default=float) + '\n')
        print(sc, bg, d, s, 'done', flush=True)


In [ ]:
%%writefile real_data.py
"""Pig tissue atlas (E-MTAB-5895): HIVE-Bic and baselines, GO enrichment and pig-to-human preservation in GTEx v10."""
import numpy as np, pandas as pd, json, os, sys, time
from scipy.stats import hypergeom
from hive import *
from baselines import METHODS
D = os.environ.get('HIVE_DATA', 'data'); OUT = os.environ.get('HIVE_OUT', 'results')

TISSUE_MAP = {'pituitary gland': 'Pituitary', 'cortex of kidney': 'Kidney', 'heart left ventricle': 'Heart', 'omentum': 'Adipose Tissue',
              'spleen': 'Spleen', 'alveolus of lung': 'Lung', 'skeletal muscle tissue': 'Muscle', 'caecum': 'Colon', 'colon': 'Colon',
              'uterus': 'Uterus', 'duodenum': 'Small Intestine', 'ileum': 'Small Intestine', 'stomach': 'Stomach',
              'hippocampal formation': 'Brain', 'amygdala': 'Brain', 'frontal cortex': 'Brain', 'occipital cortex': 'Brain',
              'brainstem': 'Brain', 'medulla oblongata': 'Brain', 'pons': 'Brain', 'cerebellum': 'Brain', 'corpus callosum': 'Brain',
              'dorsal plus ventral thalamus': 'Brain'}


def load():
    X = pd.read_csv(f'{D}/pig_counts_2000.tsv', sep='\t', index_col=0)
    meta = pd.read_csv(f'{D}/pig_samples.tsv', sep='\t', index_col=0)
    genes = pd.read_csv(f'{D}/pig_genes_2000.tsv', sep='\t')
    return X, meta, genes


class GO:
    def __init__(self, genes, min_size=10, max_size=500):
        go = pd.read_parquet(f'{D}/GeneOntologyBiologicalProcess(ToppGene).parquet', columns=['gene_id', 'concept_id', 'concept_name'])
        go['gene_id'] = pd.to_numeric(go.gene_id, errors='coerce')
        uni = set(genes.human_geneid.astype(int))
        go = go[go.gene_id.isin(uni)]
        sizes = go.groupby('concept_id').size()
        keep = sizes[(sizes >= min_size) & (sizes <= max_size)].index
        go = go[go.concept_id.isin(keep)]
        self.names = go.drop_duplicates('concept_id').set_index('concept_id').concept_name
        self.sets = go.groupby('concept_id').gene_id.apply(set)
        self.annotated = set(go.gene_id)
        self.gid = genes.human_geneid.astype(int).values

    def enrich(self, I):
        ids = set(self.gid[I]) & self.annotated
        N = len(self.annotated); n = len(ids)
        if n < 3:
            return None
        res = []
        for t, S in self.sets.items():
            k = len(ids & S)
            if k >= 2:
                res.append((t, k, len(S), hypergeom.sf(k - 1, N, len(S), n)))
        if not res:
            return None
        df = pd.DataFrame(res, columns=['term', 'k', 'size', 'p'])
        m = len(self.sets)                                   # BH over all tested terms (absent terms have p = 1)
        df = df.sort_values('p'); df['q'] = np.minimum(1, df.p * m / (np.arange(len(df)) + 1))
        df['q'] = df.q[::-1].cummin()[::-1]
        df['name'] = df.term.map(self.names)
        return df


class Preservation:
    """Pig-to-human module preservation (density and connectivity Z-scores, Zsummary) in GTEx v10."""
    def __init__(self, genes, Ypig, per_tissue=50, seed=0):
        E = pd.read_parquet(f'{D}/gtex_v10_subset.parquet')
        rng = np.random.default_rng(seed)
        idx = []
        for t, g in E.groupby('tissue_type'):
            idx += list(rng.choice(g.index, min(per_tissue, len(g)), replace=False))
        E = E.loc[idx]
        self.tissue = E.tissue_type.values
        H = E.drop(columns='tissue_type')
        cols = genes.human_ens.values
        self.has = np.array([c in H.columns for c in cols])
        Hm = np.full((len(cols), len(H)), np.nan)
        Hm[self.has] = H[cols[self.has]].values.T
        self.H = Hm
        self.Ch = np.corrcoef(np.nan_to_num(Hm))                      # human gene-gene correlation (all sampled tissues)
        self.Cp = np.corrcoef(Ypig)                                   # pig gene-gene correlation (all 51 samples)
        self.rng = rng
        self.tissues = np.array(sorted(set(self.tissue)))
        self.tprof = np.vstack([np.nanmean(Hm[:, self.tissue == t], 1) for t in self.tissues]).T

    def _stats(self, I, sign):
        Ch = self.Ch[np.ix_(I, I)] * np.outer(sign, sign); Cp = self.Cp[np.ix_(I, I)]
        iu = np.triu_indices(len(I), 1)
        dens = np.mean(Ch[iu])
        conn = np.corrcoef(Cp[iu], self.Ch[np.ix_(I, I)][iu])[0, 1]
        return dens, conn

    def zsummary(self, I, sign, B=200):
        I = np.asarray(I); ok = self.has[I]; I = I[ok]; sign = np.asarray(sign)[ok]
        if len(I) < 5:
            return dict(n_mapped=len(I), z_density=np.nan, z_conn=np.nan, zsummary=np.nan)
        d0, c0 = self._stats(I, sign)
        pool = np.where(self.has)[0]; dn, cn = [], []
        for _ in range(B):
            R = self.rng.choice(pool, len(I), replace=False)
            # null modules keep the first principal sign pattern of their own pig correlation matrix
            s = np.sign(np.linalg.eigh(self.Cp[np.ix_(R, R)])[1][:, -1]); s[s == 0] = 1
            d, c = self._stats(R, s); dn.append(d); cn.append(c)
        zd = (d0 - np.mean(dn)) / np.std(dn); zc = (c0 - np.mean(cn)) / np.std(cn)
        return dict(n_mapped=len(I), density=d0, conn=c0, z_density=zd, z_conn=zc, zsummary=(zd + zc) / 2)

    def tissue_concordance(self, I, sign, pig_tissues):
        I = np.asarray(I); ok = self.has[I]
        if ok.sum() < 3:
            return np.nan, None
        prof = self.tprof[I[ok]]
        prof = (prof - prof.mean(1, keepdims=True)) / (prof.std(1, keepdims=True) + 1e-9)
        score = (prof * np.asarray(sign)[ok][:, None]).mean(0)
        top = self.tissues[np.argmax(score)]
        mapped = {TISSUE_MAP.get(t) for t in pig_tissues} - {None}
        return float(top in mapped) if mapped else np.nan, top


def describe(b, Z, meta, genes, go, pres):
    I, J = np.asarray(b[0]), np.asarray(b[1])
    u = np.linalg.svd(Z[np.ix_(I, J)], full_matrices=False)[0][:, 0]
    sign = np.sign(u); sign[sign == 0] = 1
    if np.sum(sign * np.abs(u)) < 0: sign = -sign
    tissues = list(meta.tissue.values[J])
    e = go.enrich(I)
    top = e.iloc[0] if e is not None else None
    zs = pres.zsummary(I, sign)
    tc, htop = pres.tissue_concordance(I, sign, tissues)
    return dict(n_genes=len(I), n_cond=len(J), tissues=sorted(set(tissues)), animals=len(set(meta.individual.values[J])),
                frac_up=float(np.mean(sign > 0)), go_sig=int(e is not None and (e.q < 0.05).any()),
                go_top=None if top is None else top['name'], go_q=None if top is None else float(top['q']), go_k=None if top is None else int(top['k']),
                n_go_sig=0 if e is None else int((e.q < 0.05).sum()), human_top_tissue=htop, tissue_concord=tc,
                genes=list(genes.pig_sym.values[I][:40]), **zs)


def main(n_runs=10):
    X, meta, genes = load()
    lib = meta.lib_size.values; groups = meta.tissue.values
    go = GO(genes)
    Yfull = log_cpm(X.values, lib)
    pres = Preservation(genes, Yfull)
    rows, runs = [], []
    for r in range(n_runs):
        rng = np.random.default_rng(500 + r)
        Y1, Y2, phi = thin_folds(X.values, lib, groups, 0.5, rng)
        Z1, Z2 = standardize(Y1), standardize(Y2)
        t = time.time(); res = run_hive(X.values, lib, groups, seed=500 + r, folds=(Y1, Y2)); tt = time.time() - t
        cert = [c for c in res['candidates'] if c['certified']]
        runs.append(dict(run=r, n_cand=len(res['candidates']), n_cert=len(cert), time=tt, hv=res['hv'],
                         biclusters=[dict(I=c['I'].tolist(), J=c['J'].tolist(), T=c['T'], p=c['p'], q=c['q'], delta=c['delta_disc']) for c in cert],
                         uncert=[dict(I=c['I'].tolist(), J=c['J'].tolist(), T=c['T'], p=c['p'], q=c['q'], delta=c['delta_disc']) for c in res['candidates'] if not c['certified']]))
        for c in cert:
            d = describe((c['I'], c['J']), Z1, meta, genes, go, pres); d.update(method='HIVE-Bic', run=r, certified=1, T=c['T'], q=c['q']); rows.append(d)
        if r == 0:                                       # baselines on the same folds as run 0
            cfg = dict(DEFAULT); cfg['n_perm'] = 499
            for name, f in METHODS.items():
                t = time.time(); found = f(Y1, Z1, np.random.default_rng(500)); tt2 = time.time() - t
                cr = certify(Z2, [dict(I=I, J=J) for I, J in found], cfg, np.random.default_rng(507), Y2.mean(1))
                rej, qv = bh([c['p'] for c in cr], cfg['q'])
                for (I, J), c, k, qq in zip(found, cr, rej, qv):
                    d = describe((I, J), Z1, meta, genes, go, pres); d.update(method=name, run=0, certified=int(k), T=c['T'], q=float(qq), time=tt2); rows.append(d)
            # HIVE uncertified candidates, described for contrast
            for c in res['candidates']:
                if not c['certified']:
                    d = describe((c['I'], c['J']), Z1, meta, genes, go, pres); d.update(method='HIVE-Bic', run=0, certified=0, T=c['T'], q=c['q']); rows.append(d)
        print('run', r, 'cert', len(cert), 'time', round(tt, 1), flush=True)
        json.dump(runs, open(f'{OUT}/real_runs.json', 'w'), default=float)
        pd.DataFrame(rows).to_json(f'{OUT}/real_biclusters.json', orient='records', default_handler=str)


if __name__ == '__main__':
    main(int(sys.argv[1]) if len(sys.argv) > 1 else 10)


In [ ]:
%%writefile real_analysis.py
"""Real-data summaries: recurrence over thinning draws, baseline comparison and Fig. 9."""
import numpy as np, pandas as pd, json, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from hive import log_cpm, standardize
R = os.environ.get('HIVE_OUT', 'results'); F = os.environ.get('HIVE_FIGS', 'figs')
D = os.environ.get('HIVE_DATA', 'data')
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Liberation Serif', 'Times New Roman', 'DejaVu Serif'], 'font.size': 7,
                     'axes.linewidth': 0.6, 'mathtext.fontset': 'stix', 'savefig.dpi': 600})
NAMES = ['Neural (M1)', 'Intestinal epithelium (M2)', 'Lymphoid / B cell (M3)', 'Striated muscle (M4)']


def jac(a, b):
    a, b = set(a), set(b); return len(a & b) / len(a | b)


def main():
    runs = json.load(open(f'{R}/real_runs.json'))
    d = pd.read_json(f'{R}/real_biclusters.json')
    X = pd.read_csv(f'{D}/pig_counts_2000.tsv', sep='\t', index_col=0); meta = pd.read_csv(f'{D}/pig_samples.tsv', sep='\t', index_col=0)
    Z = standardize(log_cpm(X.values, meta.lib_size.values))
    ref = runs[0]['biclusters']
    out = dict(n_cert_runs=[r['n_cert'] for r in runs], n_cand_runs=[r['n_cand'] for r in runs], time=[r['time'] for r in runs],
               iters=[len(r['hv']) - 1 for r in runs])
    rec = []
    for b in ref:
        js = [max(jac(b['I'], c['I']) for c in runs[t]['biclusters']) for t in range(1, len(runs))]
        jc = [max(jac(b['J'], c['J']) for c in runs[t]['biclusters']) for t in range(1, len(runs))]
        rec.append(dict(n_found=int(sum(j >= 0.3 for j in js)) + 1, gene_jac=float(np.mean(js)), cond_jac=float(np.mean(jc))))
    out['recurrence'] = rec
    h = d[(d.method == 'HIVE-Bic') & (d.certified == 1)]
    out['hive_all'] = dict(n=len(h), go_sig=float(h.go_sig.mean()), z_med=float(h.zsummary.median()), z_gt2=float((h.zsummary > 2).mean()),
                           zd_med=float(h.z_density.median()), tc=float(h.tissue_concord.mean()))
    h0 = h[h.run == 0].reset_index(drop=True)
    out['run0'] = h0[['n_genes', 'n_cond', 'tissues', 'animals', 'go_top', 'go_q', 'go_k', 'n_go_sig', 'z_density', 'z_conn', 'zsummary', 'human_top_tissue', 'tissue_concord', 'q', 'T', 'genes']].to_dict('records')
    rows = []
    for m, g in d[d.run == 0].groupby('method'):
        c = g[g.certified == 1]
        rows.append(dict(method=m, n=len(g), cert=int(g.certified.sum()), go_all=float(g.go_sig.mean()),
                         go_cert=float(c.go_sig.mean()) if len(c) else np.nan, z_cert=float(c.zsummary.median()) if len(c) else np.nan,
                         tc_cert=float(c.tissue_concord.mean()) if len(c) else np.nan, z_all=float(g.zsummary.median()),
                         genes=float(g.n_genes.median()), conds=float(g.n_cond.median())))
    out['baselines'] = rows
    json.dump(out, open(f'{R}/real_summary.json', 'w'), indent=1, default=float)

    # ---- Fig. 9: tissue activity of run-0 modules and preservation statistics
    tissues = meta.tissue.values
    order = ['frontal cortex', 'occipital cortex', 'hippocampal formation', 'amygdala', 'dorsal plus ventral thalamus', 'corpus callosum', 'brainstem',
             'pons', 'medulla oblongata', 'cerebellum', 'pituitary gland', 'heart left ventricle', 'skeletal muscle tissue', 'stomach', 'duodenum', 'ileum',
             'caecum', 'colon', 'cortex of kidney', 'alveolus of lung', 'spleen', 'mesenteric lymph node', 'tonsil', 'omentum', 'uterus', 'epididymis', 'penis']
    A = np.zeros((len(ref), len(order))); Jm = np.zeros_like(A)
    for k, b in enumerate(ref):
        I, J = np.array(b['I']), np.array(b['J'])
        u = np.linalg.svd(Z[np.ix_(I, J)], full_matrices=False)[0][:, 0]
        s = np.sign(u); s[s == 0] = 1
        if np.sum(s * np.abs(u)) < 0: s = -s
        prof = (Z[I] * s[:, None]).mean(0)
        for t, name in enumerate(order):
            idx = np.where(tissues == name)[0]
            A[k, t] = prof[idx].mean(); Jm[k, t] = np.mean(np.isin(idx, J))
    fig = plt.figure(figsize=(3.45, 4.3))
    ax = fig.add_axes([0.30, 0.50, 0.66, 0.40])
    im = ax.imshow(A, cmap='Greys', aspect='auto', vmin=min(0, A.min()), vmax=A.max())
    for k in range(A.shape[0]):
        for t in range(A.shape[1]):
            if Jm[k, t] > 0:
                ax.plot(t, k, marker='o', ms=2.2, mfc='white', mec='black', mew=0.4)
    ax.set_yticks(range(len(ref))); ax.set_yticklabels(NAMES[:len(ref)], fontsize=6.5)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=90, fontsize=5.5)
    ax.set_title('(a) Module activity across pig tissues', fontsize=7.5)
    cax = fig.add_axes([0.30, 0.955, 0.66, 0.012]); cb = fig.colorbar(im, cax=cax, orientation='horizontal'); cb.ax.tick_params(labelsize=5, length=1.5, pad=1)
    cax.xaxis.set_ticks_position('top'); cb.set_label('sign-aligned mean z of module genes', fontsize=5.5, labelpad=-16)
    ax2 = fig.add_axes([0.30, 0.02, 0.66, 0.17])
    zd = [r['z_density'] for r in out['run0']]; zc = [r['z_conn'] for r in out['run0']]; zs = [r['zsummary'] for r in out['run0']]
    y = np.arange(len(zd))[::-1]; hh = 0.26
    ax2.barh(y + hh, zd, hh, color='black', label=r'$Z_{\mathrm{density}}$')
    ax2.barh(y, zc, hh, color='white', edgecolor='black', hatch='////', lw=0.5, label=r'$Z_{\mathrm{connectivity}}$')
    ax2.barh(y - hh, zs, hh, color='0.6', edgecolor='black', lw=0.5, label=r'$Z_{\mathrm{summary}}$')
    for v in (2, 10): ax2.axvline(v, ls='--', color='black', lw=0.5)
    ax2.axvline(0, color='black', lw=0.5)
    ax2.set_yticks(y); ax2.set_yticklabels(NAMES[:len(zd)], fontsize=6.5); ax2.set_xlabel('Preservation Z in GTEx v10', fontsize=7)
    ax2.set_title('(b) Pig-to-human preservation', fontsize=7.5)
    ax2.legend(frameon=False, fontsize=5.5, ncol=3, loc='lower center', bbox_to_anchor=(0.35, -0.95))
    fig.savefig(f'{F}/Fig9_real.png', bbox_inches='tight'); plt.close(fig)
    print(json.dumps({k: out[k] for k in ('recurrence', 'hive_all', 'n_cert_runs')}, default=float))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile analysis.py
"""Statistics and black-and-white figures for the HIVE-Bic experiments."""
import numpy as np, pandas as pd, json, glob, os, sys
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

R = os.environ.get('HIVE_OUT', 'results'); F = os.environ.get('HIVE_FIGS', 'figs')
os.makedirs(F, exist_ok=True)
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Liberation Serif', 'Times New Roman', 'DejaVu Serif'],
                     'font.size': 8, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
                     'axes.labelsize': 8, 'legend.fontsize': 7, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
                     'mathtext.fontset': 'stix', 'savefig.dpi': 600})
ORDER = ['HIVE-Bic', 'LAS', 'SSVD', 'SpecCo', 'SpecBic', 'ISA', 'Plaid', 'CC', 'BPSO', 'LBBSO']
LABEL = {'LBBSO': 'LB-BSO', 'SpecCo': 'Spec-Co', 'SpecBic': 'Spec-Bic'}
lab = lambda m: LABEL.get(m, m)
NS = 12


def load(exp):
    rows = [json.loads(l) for f in glob.glob(f'{R}/{exp}_part*.jsonl') for l in open(f)]
    d = pd.DataFrame(rows)
    if len(d) == 0:
        return d
    d['scen'] = np.where(d.background == 'real', 'real-2.0', 'nb-' + d.delta.astype(str))
    base = {'main': None, 'null': 3000, 'ablation': 1030, 'sens': 4000}[exp]
    if exp == 'main':
        d = d[((d.background == 'nb') & (d.seed < 1000 + NS)) | ((d.background == 'real') & (d.seed < 2000 + NS))]
    return d


def flat(df):
    df = df.copy(); df.columns = ['_'.join([c for c in col if c]) if isinstance(col, tuple) else col for col in df.columns]; return df


def ci_mean(x, B=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    bs = rng.choice(x, (B, len(x))).mean(1)
    return tuple(np.percentile(bs, [2.5, 97.5]))


def clopper(k, n, a=0.05):
    lo = stats.beta.ppf(a / 2, k, n - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(1 - a / 2, k + 1, n - k) if k < n else 1.0
    return lo, hi


def main_tables(d):
    out = {}
    d = d.assign(fdp=np.nan_to_num(d.false_cert.values.astype(float)), fdp_raw=np.nan_to_num(d.false_out.values.astype(float)))
    g = d.groupby(['scen', 'method'])
    for metric in ['fdp', 'fdp_raw', 'rec', 'rel', 'rec_cert', 'rel_cert', 'n_out', 'n_cert', 'false_cert', 'time', 'recc_shift', 'recc_scale', 'recc_anti']:
        out[metric] = g[metric].agg(['mean', 'std']).reset_index()
    return out


def friedman(d, metric):
    d = d[d.background == 'nb']
    P = d.pivot_table(index=['scen', 'seed'], columns='method', values=metric)[ORDER].dropna()
    ranks = P.rank(axis=1, ascending=False).mean()
    chi, p = stats.friedmanchisquare(*[P[c].values for c in ORDER])
    k, N = len(ORDER), len(P)
    q05 = {10: 3.164}[k]                     # Studentized range / sqrt(2), alpha = 0.05, k = 10 (Demsar 2006)
    cd = q05 * np.sqrt(k * (k + 1) / (6 * N))
    return ranks.sort_values(), chi, p, cd, N


def wilcoxon_vs(d, metric, ref='HIVE-Bic'):
    d = d[d.background == 'nb']
    P = d.pivot_table(index=['scen', 'seed'], columns='method', values=metric).dropna()
    res = []
    for m in ORDER[1:]:
        x, y = P[ref].values, P[m].values; diff = x - y
        nz = diff[diff != 0]
        if len(nz) == 0:
            res.append((m, 0.0, 1.0, 0.0, 0.0, 0.0)); continue
        w = stats.wilcoxon(x, y, zero_method='wilcox')
        rp = stats.rankdata(np.abs(nz)); rbc = (rp[nz > 0].sum() - rp[nz < 0].sum()) / rp.sum()
        lo, hi = ci_mean(diff)
        res.append((m, diff.mean(), w.pvalue, rbc, lo, hi))
    r = pd.DataFrame(res, columns=['method', 'diff', 'p', 'rbc', 'lo', 'hi'])
    order = np.argsort(r.p.values); adj = np.empty(len(r)); run = 0
    for i, j in enumerate(order):
        run = max(run, min(1, (len(r) - i) * r.p.values[j])); adj[j] = run
    r['p_holm'] = adj
    return r


# ------------------------------------------------------------------ figures
def fig_null(n):
    g = n.groupby('method')
    meth = ['HIVE-Bic', 'w/o thinning'] + [m for m in ORDER[1:] if m in set(n.method)]
    frac = [g.get_group(m).any_false_cert.mean() for m in meth]
    k = [g.get_group(m).any_false_cert.sum() for m in meth]; N = [len(g.get_group(m)) for m in meth]
    cis = [clopper(a, b) for a, b in zip(k, N)]
    nout = [g.get_group(m).n_out.mean() for m in meth]
    fig, ax = plt.subplots(2, 1, figsize=(3.45, 4.6), gridspec_kw=dict(hspace=0.95))
    x = np.arange(len(meth))
    ax[0].bar(x, frac, color=['black', '0.45'] + ['white'] * (len(meth) - 2), edgecolor='black', linewidth=0.6, width=0.7)
    ax[0].errorbar(x, frac, yerr=[np.array(frac) - [c[0] for c in cis], [c[1] for c in cis] - np.array(frac)], fmt='none', ecolor='black', capsize=2, lw=0.6)
    ax[0].axhline(0.10, ls='--', color='black', lw=0.7); ax[0].text(len(meth) - 0.6, 0.115, r'$q=0.10$', ha='right', fontsize=7)
    ax[0].set_ylabel('Matrices with a certified block'); ax[0].set_ylim(0, 1.05)
    ax[0].set_xticks(x); ax[0].set_xticklabels([lab(m) for m in meth], rotation=45, ha='right')
    ax[0].set_title('(a) Empirical FDR on structure-free counts', fontsize=8)
    ax[1].bar(x, nout, color='white', edgecolor='black', hatch='////', linewidth=0.6, width=0.7, label='reported')
    ax[1].bar(x, [g.get_group(m).n_cert.mean() for m in meth], color='black', width=0.7, label='certified')
    ax[1].set_ylabel('Blocks per null matrix'); ax[1].set_xticks(x); ax[1].set_xticklabels([lab(m) for m in meth], rotation=45, ha='right')
    ax[1].legend(frameon=False, loc='upper left', bbox_to_anchor=(0.0, 1.0)); ax[1].set_title('(b) Blocks reported and certified', fontsize=8)
    ax[1].set_ylim(0, max(nout) * 1.25)
    fig.savefig(f'{F}/Fig2_null.png', bbox_inches='tight'); plt.close(fig)


def fig_main(d):
    scen = ['nb-1.5', 'nb-2.0', 'nb-2.5']
    sl = {'nb-1.5': r'NB, $\delta=1.5$', 'nb-2.0': r'NB, $\delta=2.0$', 'nb-2.5': r'NB, $\delta=2.5$', 'real-2.0': r'Atlas, $\delta=2.0$'}
    mk = ['o', 's', '^', 'D']; fc = ['black', 'white', '0.55', 'white']
    fig, ax = plt.subplots(2, 1, figsize=(3.45, 5.6), sharex=False, gridspec_kw=dict(hspace=0.32))
    y = np.arange(len(ORDER))[::-1]
    for a, metric, title in ((ax[0], 'rec_cert', '(a) Recovery of certified blocks'), (ax[1], 'rel_cert', '(b) Relevance of certified blocks')):
        for s, (sc, m_, f_) in enumerate(zip(scen, mk, fc)):
            sub = d[d.scen == sc].groupby('method')[metric]
            mu = np.array([sub.mean().get(m, np.nan) for m in ORDER]); se = np.array([sub.std().get(m, np.nan) / np.sqrt(sub.count().get(m, 1)) for m in ORDER])
            a.errorbar(np.nan_to_num(mu), y + (s - 1) * 0.22, xerr=1.96 * np.nan_to_num(se), fmt=m_, mfc=f_, mec='black', ecolor='black', ms=3.5, lw=0.6, capsize=1.5, label=sl[sc])
        a.set_title(title, fontsize=8); a.set_xlim(-0.02, 1.0); a.grid(axis='x', color='0.85', lw=0.4)
        for yy in y[:-1]: a.axhline(yy - 0.5, color='0.9', lw=0.4)
    for a in ax: a.set_yticks(y); a.set_yticklabels([lab(m) for m in ORDER])
    ax[0].set_xlabel('Recovery'); ax[1].set_xlabel('Relevance')
    h, l = ax[0].get_legend_handles_labels()
    fig.legend(h, l, ncol=3, frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.005))
    fig.subplots_adjust(bottom=0.12)
    fig.savefig(f'{F}/Fig3_main.png', bbox_inches='tight'); plt.close(fig)


def fig_pattern(d):
    M = np.array([[d[d.method == m][f'recc_{p}'].mean() for p in ('shift', 'scale', 'anti')] for m in ORDER])
    fig, ax = plt.subplots(figsize=(3.3, 3.0))
    im = ax.imshow(M, cmap='Greys', vmin=0, vmax=max(0.8, M.max()), aspect='auto')
    for i in range(M.shape[0]):
        for j in range(3):
            ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white' if M[i, j] > 0.45 else 'black')
    ax.set_xticks(range(3)); ax.set_xticklabels(['Shift', 'Shift-scale', 'Anti-correlated'])
    ax.set_yticks(range(len(ORDER))); ax.set_yticklabels([lab(m) for m in ORDER])
    cb = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.04); cb.set_label('Certified recovery', fontsize=7); cb.ax.tick_params(labelsize=6)
    fig.savefig(f'{F}/Fig4_patterns.png', bbox_inches='tight'); plt.close(fig)


def fig_cd(ranks, cd, fname, title):
    k = len(ranks); lo, hi = 1, k
    fig, ax = plt.subplots(figsize=(3.45, 1.9))
    ax.set_xlim(lo - 0.3, hi + 0.3); ax.set_ylim(0, 1); ax.axis('off')
    ax.plot([lo, hi], [0.8, 0.8], color='black', lw=0.7)
    for t in range(lo, hi + 1):
        ax.plot([t, t], [0.8, 0.83], color='black', lw=0.6); ax.text(t, 0.86, str(t), ha='center', fontsize=7)
    ax.plot([lo, lo + cd], [0.97, 0.97], color='black', lw=1.2); ax.text(lo + cd / 2, 0.99, f'CD = {cd:.2f}', ha='center', va='bottom', fontsize=7)
    items = list(ranks.items()); half = (k + 1) // 2
    for i, (m, r) in enumerate(items):
        left = i < half
        yy = 0.68 - 0.105 * (i if left else (k - 1 - i))
        xe = lo - 0.25 if left else hi + 0.25
        ax.plot([r, r, xe], [0.8, yy, yy], color='black', lw=0.5)
        ax.text(xe + (-0.05 if left else 0.05), yy, f'{lab(m)} ({r:.2f})', ha='right' if left else 'left', va='center', fontsize=6.5)
    # cliques: groups of methods whose rank range is below CD
    vals = ranks.values; cl = []
    for i in range(k):
        j = i
        while j + 1 < k and vals[j + 1] - vals[i] < cd: j += 1
        if j > i and not any(a <= i and j <= b for a, b in cl): cl.append((i, j))
    for c, (i, j) in enumerate(cl):
        yb = 0.75 - 0.035 * c
        ax.plot([vals[i] - 0.03, vals[j] + 0.03], [yb, yb], color='black', lw=1.6)
    fig.savefig(f'{F}/{fname}', bbox_inches='tight'); plt.close(fig)


def fig_ablation(a):
    var = ['HIVE-Bic', 'w/o thinning', 'w/o search-adjusted null', 'w/o diversity', 'w/o LogitBoost', 'in-sample LogitBoost', 'w/o polish', 'Pearson objective']
    g = a.groupby('method')
    rc = [g.get_group(v).rec_cert.mean() for v in var]; rl = [g.get_group(v).rel_cert.mean() for v in var]
    rce = [1.96 * g.get_group(v).rec_cert.std() / np.sqrt(len(g.get_group(v))) for v in var]
    rle = [1.96 * g.get_group(v).rel_cert.std() / np.sqrt(len(g.get_group(v))) for v in var]
    fig, ax = plt.subplots(figsize=(3.45, 2.7))
    y = np.arange(len(var))[::-1]; h = 0.36
    ax.barh(y + h / 2, rc, h, xerr=rce, color='black', ecolor='0.4', error_kw=dict(lw=0.6, capsize=1.5), label='Recovery')
    ax.barh(y - h / 2, rl, h, xerr=rle, color='white', edgecolor='black', hatch='////', lw=0.6, ecolor='0.4', error_kw=dict(lw=0.6, capsize=1.5), label='Relevance')
    ax.set_yticks(y); ax.set_yticklabels(var); ax.set_xlabel('Certified score'); ax.set_xlim(0, 1)
    ax.legend(frameon=False, loc='lower right'); ax.grid(axis='x', color='0.88', lw=0.4)
    fig.savefig(f'{F}/Fig6_ablation.png', bbox_inches='tight'); plt.close(fig)


def fig_sens(s):
    groups = [('n_bees', r'Swarm size $N$'), ('lb_rounds', r'Boosting rounds $M$'), ('eps', r'Thinning fraction $\varepsilon$'), ('lam', r'Diversity weight $\lambda$'), ('tau_g', r'Gene threshold $\tau_g$')]
    fig, axs = plt.subplots(3, 2, figsize=(3.45, 4.4), gridspec_kw=dict(wspace=0.12, hspace=0.75))
    axs = axs.ravel(); axs[5].axis('off')
    for ax, (p, t) in zip(axs, groups):
        sub = s[s.method.str.startswith(p + '=')].copy(); sub['v'] = sub.method.str.split('=').str[1].astype(float)
        gg = sub.groupby('v')
        v = np.array(sorted(gg.groups))
        for metric, mk, fc in (('rec_cert', 'o', 'black'), ('rel_cert', 's', 'white')):
            mu = gg[metric].mean().loc[v].values; se = (gg[metric].std() / np.sqrt(gg[metric].count())).loc[v].values
            ax.errorbar(range(len(v)), mu, yerr=1.96 * se, fmt='-' + mk, color='black', mfc=fc, ms=3.5, lw=0.7, capsize=1.5, label='Recovery' if metric == 'rec_cert' else 'Relevance')
        ax.set_xticks(range(len(v))); ax.set_xticklabels([f'{x:g}' for x in v]); ax.set_xlabel(t); ax.set_ylim(0, 1)
        ax.grid(axis='y', color='0.88', lw=0.4)
    for k in (0, 2, 4): axs[k].set_ylabel('Certified score')
    for k in (1, 3): axs[k].set_yticklabels([])
    h, l = axs[0].get_legend_handles_labels(); axs[5].legend(h, l, frameon=False, loc='center')
    fig.savefig(f'{F}/Fig7_sensitivity.png', bbox_inches='tight'); plt.close(fig)


def fig_conv(d):
    h = d[d.method == 'HIVE-Bic']
    T = max(len(x) for x in h.hv)
    fig, ax = plt.subplots(2, 1, figsize=(3.45, 4.4), gridspec_kw=dict(hspace=0.75))
    for sc, ls in (('nb-1.5', ':'), ('nb-2.0', '--'), ('nb-2.5', '-'), ('real-2.0', '-.')):
        H = np.full((0, T), np.nan)
        for x in h[h.scen == sc].hv:
            x = np.asarray(x, float); x = x / x[-1] if x[-1] > 0 else x
            row = np.full(T, np.nan); row[:len(x)] = x; row[len(x):] = x[-1]; H = np.vstack([H, row])
        ax[0].plot(np.arange(T), np.nanmedian(H, 0), ls=ls, color='black', lw=0.9, label=sc.replace('nb-', r'NB $\delta=$').replace('real-2.0', r'Atlas $\delta=2.0$'))
    ax[0].set_xlabel('Iteration $t$'); ax[0].set_ylabel(r'$\mathrm{HV}_t/\mathrm{HV}_{\mathrm{final}}$ (median)'); ax[0].legend(frameon=False, loc='lower right')
    ax[0].set_title('(a) Archive hypervolume', fontsize=8); ax[0].grid(color='0.9', lw=0.4)
    tm = d.groupby('method').time.median().reindex(ORDER)
    x = np.arange(len(ORDER))
    ax[1].bar(x, tm.values, color=['black'] + ['white'] * (len(ORDER) - 1), edgecolor='black', lw=0.6)
    ax[1].set_yscale('log'); ax[1].set_ylabel('Median run time (s)'); ax[1].set_xticks(x); ax[1].set_xticklabels([lab(m) for m in ORDER], rotation=45, ha='right')
    ax[1].set_title('(b) Run time per matrix', fontsize=8)
    fig.savefig(f'{F}/Fig8_convergence.png', bbox_inches='tight'); plt.close(fig)


if __name__ == '__main__':
    d = load('main'); res = {}
    if len(d):
        fig_main(d); fig_pattern(d[d.background == 'nb'])
        for metric in ('rec_cert', 'rel_cert'):
            ranks, chi, p, cd, N = friedman(d, metric)
            res[f'friedman_{metric}'] = dict(ranks=ranks.to_dict(), chi=chi, p=p, cd=cd, N=N)
            fig_cd(ranks, cd, f'Fig5_cd_{metric}.png', metric)
            res[f'wilcoxon_{metric}'] = wilcoxon_vs(d, metric).to_dict('records')
        for metric in ('recc_shift', 'recc_scale', 'recc_anti'):
            res[f'wilcoxon_{metric}'] = wilcoxon_vs(d, metric).to_dict('records')
        res['main'] = {k: v.to_dict('records') for k, v in main_tables(d).items()}
        fig_conv(d)
        h = d[d.method == 'HIVE-Bic'].assign(fdp=lambda x: np.nan_to_num(x.false_cert.values.astype(float)))
        res['hive_fdp_ci'] = {sc: (float(g.fdp.mean()),) + ci_mean(g.fdp.values) for sc, g in h.groupby('scen')}
        res['fdp_all'] = {m: float(np.nan_to_num(g.false_cert.values.astype(float)).mean()) for m, g in d[d.background == 'nb'].groupby('method')}
    n = load('null')
    if len(n):
        fig_null(n)
        res['null'] = flat(n.groupby('method')[['n_out', 'n_cert', 'any_false_cert']].agg(['mean', 'sum', 'count']).reset_index()).to_dict('records')
    a = load('ablation')
    if len(a):
        fig_ablation(a); a = a.assign(fdp=np.nan_to_num(a.false_cert.values.astype(float))); res['ablation'] = flat(a.groupby('method')[['rec', 'rel', 'rec_cert', 'rel_cert', 'n_out', 'n_cert', 'fdp', 'time', 'n_eval', 'recc_shift', 'recc_scale', 'recc_anti']].agg(['mean', 'std']).reset_index()).to_dict('records')
        res['ablation_w'] = {}
        for metric in ('rec_cert', 'rel_cert'):
            P = a.pivot_table(index='seed', columns='method', values=metric)
            res['ablation_w'][metric] = {m: float(stats.wilcoxon(P['HIVE-Bic'], P[m]).pvalue) if (P['HIVE-Bic'] != P[m]).any() else 1.0 for m in P.columns if m != 'HIVE-Bic'}
    s = load('sens')
    if len(s):
        fig_sens(s); res['sens'] = flat(s.groupby('method')[['rec_cert', 'rel_cert', 'n_cert', 'time']].agg(['mean', 'std']).reset_index()).to_dict('records')
    json.dump(res, open(f'{R}/summary.json', 'w'), default=float, indent=1)
    print('ok')


In [ ]:
%%writefile fig_arch.py
"""Fig. 1: HIVE-Bic architecture (grayscale, column width)."""
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import sys, os

OUT = sys.argv[1] if len(sys.argv) > 1 else os.path.join(os.environ.get('HIVE_FIGS', 'figs'), 'Fig1_architecture.png')
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Liberation Serif', 'Times New Roman', 'DejaVu Serif'], 'mathtext.fontset': 'stix'})
fig, ax = plt.subplots(figsize=(3.5, 5.6), dpi=600)
ax.set_xlim(0, 10); ax.set_ylim(0, 16); ax.axis('off')
FS = 5.4
boxes = {}

def box(key, x, y, w, h, text, fill='white', lw=0.6, ls='-', bold=False):
    p = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.18', fc=fill, ec='black', lw=lw, ls=ls)
    ax.add_patch(p)
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=FS, fontweight='bold' if bold else 'normal', linespacing=1.15)
    boxes[key] = (x, y, w, h)

def pt(key, side, frac=0.5):
    x, y, w, h = boxes[key]
    return {'top': (x + frac * w, y + h), 'bottom': (x + frac * w, y), 'left': (x, y + frac * h), 'right': (x + w, y + frac * h)}[side]

def arrow(p0, p1, ls='-', cs='arc3', lw=0.7):
    a = FancyArrowPatch(p0, p1, arrowstyle='-|>', mutation_scale=6, lw=lw, ls=ls, color='black', connectionstyle=cs, shrinkA=0, shrinkB=0)
    ax.add_patch(a)

# shared trunk
box('in', 0.4, 14.55, 9.2, 1.15, 'RNA-seq counts $X\\in\\mathbb{N}^{n\\times m}$ (genes $\\times$ tissue samples),\nlibrary sizes $L_j$, replicate groups', fill='#E6E6E6', bold=False)
box('disp', 0.4, 13.05, 9.2, 0.95, 'Dispersion $\\hat\\phi_g$: pooled within-tissue moments, shrunk on log scale')
box('thin', 0.4, 11.55, 9.2, 0.95, 'Negative-binomial data thinning ($\\varepsilon=0.5$): $X=X^{(1)}+X^{(2)}$', fill='#E6E6E6')
# left: discovery
box('z1', 0.4, 9.85, 4.3, 1.05, 'Discovery fold $X^{(1)}$\nlog-CPM, robust row scaling $\\to Z^{(1)}$')
box('sw', 0.4, 7.75, 4.3, 1.55, 'Binary bee swarm\n(employed / onlooker / scout)\nsigmoid transfer + polish\nobjective $S(I,J)$: search-adjusted $z$')
box('ar', 0.4, 6.15, 4.3, 1.05, 'Elite archive: redundancy filter,\nhypervolume stopping rule')
box('lb', 0.4, 4.45, 4.3, 1.15, 'Cross-fitted LogitBoost\nmembership of genes and\nconditions (out-of-fold $\\hat\\pi$)')
box('co', 0.4, 2.95, 4.3, 0.95, 'Consolidation $\\to$ candidates $(I,J)$')
# right: confirmation
box('z2', 5.3, 9.85, 4.3, 1.05, 'Confirmation fold $X^{(2)}$\nlog-CPM, robust row scaling $\\to Z^{(2)}$')
box('st', 5.3, 2.95, 4.3, 0.95, 'Statistic $T(I,J)$ on $Z^{(2)}$')
box('pn', 5.3, 1.6, 4.3, 0.85, 'Column + matched-gene\npermutation nulls, $p=\\max(p_{\\mathrm{col}},p_{\\mathrm{row}})$')
box('bh', 5.3, 0.25, 4.3, 0.85, 'Benjamini$-$Hochberg ($q=0.10$)\n$\\to$ certified biclusters', fill='#E6E6E6')
box('val', 0.4, 0.25, 4.3, 2.1, 'Biological checks\n(independent of search):\nGO enrichment; pig$\\to$human\npreservation in GTEx v10\n($Z_{\\mathrm{summary}}$), tissue concordance', fill='white', ls='--')

arrow(pt('in', 'bottom'), pt('disp', 'top'))
arrow(pt('disp', 'bottom'), pt('thin', 'top'))
arrow(pt('thin', 'bottom', 0.2337), pt('z1', 'top'))
arrow(pt('thin', 'bottom', 0.7663), pt('z2', 'top'))
arrow(pt('z1', 'bottom'), pt('sw', 'top'))
arrow(pt('sw', 'bottom', 0.35), pt('ar', 'top', 0.35))
# dashed feedback: archive -> swarm (leaders, tabu anchors)
arrow(pt('ar', 'top', 0.72), pt('sw', 'bottom', 0.72), ls='--')
ax.text(pt('ar', 'top', 0.72)[0] + 0.12, (pt('ar', 'top')[1] + pt('sw', 'bottom')[1]) / 2, 'leaders,\ntabu', fontsize=4.6, va='center', ha='left', style='italic')
arrow(pt('ar', 'bottom'), pt('lb', 'top'))
arrow(pt('lb', 'bottom'), pt('co', 'top'))
arrow(pt('co', 'right'), pt('st', 'left'))
arrow(pt('z2', 'bottom'), pt('st', 'top'))
ax.text(pt('z2', 'bottom')[0] + 0.15, 6.4, 'held out from\nsearch and\nrefinement', fontsize=4.8, ha='left', va='center', style='italic')
arrow(pt('st', 'bottom'), pt('pn', 'top'))
arrow(pt('pn', 'bottom'), pt('bh', 'top'))
arrow(pt('bh', 'left'), pt('val', 'right', 0.135))
fig.savefig(OUT, dpi=600, bbox_inches='tight', pad_inches=0.02)
print('saved', OUT)


In [ ]:
%%writefile fig_nulls.py
"""Fig. 10: permutation nulls of the certification statistic, double-dipped versus held-out, on a null matrix and on the atlas."""
import numpy as np, pandas as pd, json, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from hive import *
from synth import make_matrix
D = os.environ.get('HIVE_DATA', 'data'); R = os.environ.get('HIVE_OUT', 'results'); F = os.environ.get('HIVE_FIGS', 'figs')
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Liberation Serif', 'Times New Roman', 'DejaVu Serif'], 'font.size': 7,
                     'axes.linewidth': 0.6, 'mathtext.fontset': 'stix', 'savefig.dpi': 600})


def nulls(Z, I, J, rng, bins, B=499):
    m = Z.shape[1]
    col = np.array([coherence(Z, I, rng.choice(m, len(J), replace=False)) for _ in range(B)])
    ub, cnt = np.unique(bins[I], return_counts=True); pools = {b: np.where(bins == b)[0] for b in ub}
    row = np.array([coherence(Z, np.concatenate([rng.choice(pools[b], k, replace=False) for b, k in zip(ub, cnt)]), J) for _ in range(B)])
    return col, row


def main():
    X = pd.read_csv(f'{D}/pig_counts_2000.tsv', sep='\t', index_col=0); meta = pd.read_csv(f'{D}/pig_samples.tsv', sep='\t', index_col=0)
    lib = meta.lib_size.values; groups = meta.tissue.values
    phi = estimate_dispersion(X.values, lib, groups); base = (X.values / size_factors(lib)).mean(1)
    rng = np.random.default_rng(3000)
    Xs, _ = make_matrix(base, phi, lib, rng, 0.0)
    Y1, Y2, _ = thin_folds(Xs, lib, groups, 0.5, rng)
    res = run_hive(Xs, lib, groups, seed=3000, folds=(Y1, Y2))
    Z1, Z2 = standardize(Y1), standardize(Y2)
    c = max(res['candidates'], key=lambda c: c['delta_disc'])
    I, J = c['I'], c['J']
    r = np.random.default_rng(1)
    b1 = np.digitize(Y1.mean(1), np.quantile(Y1.mean(1), np.linspace(0, 1, 11)[1:-1]))
    b2 = np.digitize(Y2.mean(1), np.quantile(Y2.mean(1), np.linspace(0, 1, 11)[1:-1]))
    col1, row1 = nulls(Z1, I, J, r, b1); col2, row2 = nulls(Z2, I, J, r, b2)
    t1, t2 = coherence(Z1, I, J), coherence(Z2, I, J)
    runs = json.load(open(f'{R}/real_runs.json'))
    Xr = X.values; rr = np.random.default_rng(500)
    Yr1, Yr2, _ = thin_folds(Xr, lib, groups, 0.5, rr)
    Zr2 = standardize(Yr2)
    br = np.digitize(Yr2.mean(1), np.quantile(Yr2.mean(1), np.linspace(0, 1, 11)[1:-1]))
    bm = runs[0]['biclusters'][3]
    Im, Jm = np.array(bm['I']), np.array(bm['J'])
    colm, rowm = nulls(Zr2, Im, Jm, np.random.default_rng(2), br)
    tm = coherence(Zr2, Im, Jm)
    out = dict(null_block=dict(n_genes=len(I), n_cond=len(J), T_disc=t1, T_conf=t2, p_disc=float(max((1 + (col1 >= t1).sum()) / 500, (1 + (row1 >= t1).sum()) / 500)),
                               p_conf=float(max((1 + (col2 >= t2).sum()) / 500, (1 + (row2 >= t2).sum()) / 500)), S=c['delta_disc']),
               muscle=dict(T=tm, max_col=float(colm.max()), max_row=float(rowm.max())))
    json.dump(out, open(f'{R}/fig10_nulls.json', 'w'), indent=1, default=float)
    fig, ax = plt.subplots(3, 1, figsize=(3.45, 4.6), gridspec_kw=dict(hspace=1.0))
    panels = [(ax[0], col1, row1, t1, '(a) Null matrix, block tested on its discovery fold'),
              (ax[1], col2, row2, t2, '(b) Same block tested on the confirmation fold'),
              (ax[2], colm, rowm, tm, '(c) Atlas muscle module (M4), confirmation fold')]
    for a, cn, rn, t, title in panels:
        lo = min(cn.min(), rn.min(), t); hi = max(cn.max(), rn.max(), t)
        bins = np.linspace(lo, hi + 1e-9, 40)
        a.hist(cn, bins=bins, color='0.55', edgecolor='black', lw=0.3, label='condition null')
        a.hist(rn, bins=bins, histtype='step', color='black', lw=0.9, hatch='////', label='gene null')
        a.axvline(t, color='black', lw=1.2, ls='--')
        a.text(t, a.get_ylim()[1] * 1.01, r'$T_{\mathrm{obs}}$', ha='center', va='bottom', fontsize=6.5, clip_on=False)
        a.set_title(title, fontsize=7.5, pad=10); a.set_ylabel('Count'); a.set_xlabel(r'Edge-corrected share $T$')
    ax[0].legend(frameon=False, fontsize=6, loc='upper center', bbox_to_anchor=(0.45, 1.0))
    fig.savefig(f'{F}/Fig10_nulls.png', bbox_inches='tight'); plt.close(fig)
    print(out)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile fig_nulltable.py
"""Fig. 2: search-adjusted null level of the rank-one share versus the random-matrix edge for a fixed block."""
import numpy as np, json, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from hive import NullTable
F = os.environ.get('HIVE_FIGS', 'figs'); R = os.environ.get('HIVE_OUT', 'results')
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Liberation Serif', 'Times New Roman', 'DejaVu Serif'], 'font.size': 7,
                     'axes.linewidth': 0.6, 'mathtext.fontset': 'stix', 'savefig.dpi': 600})
nt = NullTable(2000, 51, np.random.default_rng(12345))
fig, ax = plt.subplots(figsize=(3.45, 2.5))
out = {}
for l, ls, mk in ((5, '-', 'o'), (10, '--', 's'), (22, ':', '^')):
    ks = nt.ks.astype(int)
    mu = np.array([nt(k, l)[0] for k in ks]); sd = np.array([nt(k, l)[1] for k in ks])
    edge = (1 / np.sqrt(ks) + 1 / np.sqrt(l)) ** 2
    ax.errorbar(ks, mu, yerr=sd, fmt=ls + mk, color='black', mfc='white', ms=3, lw=0.8, capsize=1.5, label=rf'searched, $|J|={l}$')
    ax.plot(ks, edge, ls, color='0.6', lw=1.2)
    out[l] = dict(k=ks.tolist(), mu=mu.tolist(), sd=sd.tolist(), edge=edge.tolist())
ax.plot([], [], color='0.6', lw=1.2, label='random-matrix edge, fixed block')
ax.set_xscale('log'); ax.set_xlabel(r'Number of genes $|I|$'); ax.set_ylabel(r'Rank-one share $\rho$ on noise')
ax.legend(frameon=False, fontsize=6, loc='upper right'); ax.grid(color='0.9', lw=0.4)
fig.savefig(f'{F}/Fig_nulltable.png', bbox_inches='tight'); plt.close(fig)
json.dump(out, open(f'{R}/nulltable.json', 'w'), indent=1)
print({l: (round(v['mu'][0], 3), round(v['edge'][0], 3), round(v['mu'][-1], 3), round(v['edge'][-1], 3)) for l, v in out.items()})


## 2. Data
The raw counts come from the Expression Atlas FTP site. If the download is blocked, fetch the raw counts of E-MTAB-5895 by hand from the Expression Atlas Downloads tab and upload the file into `hive/data/`.

In [ ]:
!python download_data.py
!python data_prep.py
!python gtex_fetch.py

## 3. A single semi-synthetic run

In [ ]:
import numpy as np, pandas as pd
from hive import *
from synth import *
D = os.environ["HIVE_DATA"]
X = pd.read_csv(f"{D}/pig_counts_2000.tsv", sep="\t", index_col=0); meta = pd.read_csv(f"{D}/pig_samples.tsv", sep="\t", index_col=0)
lib = meta.lib_size.values; groups = meta.tissue.values
phi = estimate_dispersion(X.values, lib, groups); base = (X.values / size_factors(lib)).mean(1)
rng = np.random.default_rng(7)
Xs, truth = make_matrix(base, phi, lib, rng, 2.0)
res = run_hive(Xs, lib, groups, seed=7)
for c in res["candidates"]:
    print(len(c["I"]), "genes x", len(c["J"]), "samples | T =", round(c["T"], 3), "| p =", round(c["p"], 4), "| certified:", c["certified"])
found = [(c["I"], c["J"]) for c in res["candidates"] if c["certified"]]
print("recovery / relevance:", match_scores(found, truth)[:2])

## 4. Experiments
Resumable: re-running a cell skips finished jobs.

In [ ]:
for exp in ("main", "null", "ablation", "sens"):
    !python run_synth.py {exp} 0 1

In [ ]:
!python real_data.py 10
!python real_analysis.py

## 5. Statistics and figures

In [ ]:
!python analysis.py
!python fig_arch.py
!python fig_nulls.py
!python fig_nulltable.py
import glob
from IPython.display import Image, display
for f in sorted(glob.glob("figs/*.png")):
    print(f); display(Image(f, width=700))